# 03 v2 — Nested Out-of-Fold Model Laboratory (Diagnostic Repair)


> **v2 diagnostic repair.** This revision preserves the frozen split, feature, and modeling contracts,
> and therefore reuses valid standard-mode checkpoints. It fixes two diagnostics exposed by the first
> standard run under pandas 3 / XGBoost 3: writable NumPy copies for fold-local feature selection and
> native XGBoost TreeSHAP contributions for explainability. It does not load the locked benchmark,
> change model-selection rules, or alter the candidate feature universe.

This notebook is the **model-selection, calibration, dimensionality-control, and ensemble laboratory** for the March Machine Learning Mania 2026 project.

It begins only after notebooks `00`, `01`, and `02` have completed. Notebook `02` intentionally generated a broad candidate bank rather than selecting a model. The present notebook turns that bank into scientifically defensible development predictions under the frozen season-level folds.

## Non-negotiable scientific rules

1. **Development outcomes only.** Seasons 2022–2025 remain locked and are never scored, displayed, tuned against, or used to select a recipe here.
2. **Whole-season validation.** A tournament season is indivisible. There is no shuffled row-level cross-validation.
3. **Strict temporal order.** Every training season precedes its validation season.
4. **Nested decisions.** Imputation, scaling, feature selection, hyperparameter tuning, early-stopping length, calibration, and ensemble weights are learned from inner folds only.
5. **No outer-fold early stopping.** The outer validation season is never used to choose boosting rounds. The number of rounds is inherited from inner folds.
6. **Separate primary systems.** Men's and women's models are primary; a pooled common-feature model and partial-pooling blend are challengers.
7. **Probability quality first.** Brier score is primary. Calibration, season stability, and uncertainty are reported alongside discrimination.
8. **Bounded dimensionality.** The feature store may be broad, but each fitted model receives a fold-fitted, block-balanced, correlation-pruned subset.
9. **RAM-safe execution.** The 2026 Stage 2 matrix is not loaded in this notebook. Historical columns are projected from Parquet, downcast to `float32`, trained sequentially, checkpointed, and garbage-collected.
10. **Restartability.** Every completed outer-fold/model result is cached atomically. Re-running the notebook resumes rather than starting over.

## Model ladder

The standard run evaluates:

- constant 0.50;
- seed-only logistic regression;
- corrected Elo + seed logistic regression;
- compact and rich elastic-net logistic regression;
- histogram gradient boosting;
- XGBoost and LightGBM direct classifiers;
- ridge, XGBoost, and LightGBM point-margin regressors;
- cross-fitted probability calibration;
- constrained nonnegative ensembles;
- separate, pooled, and partial-pooling architectures.

A small regularized PyTorch multilayer perceptron is implemented as an **optional exhaustive challenger**, not assumed to be superior to tabular methods.

## Resource modes

- `smoke`: verifies the complete machinery on the latest fold of each context.
- `standard`: runs all frozen development folds with bounded nested search and strict feature caps.
- `exhaustive`: expands search, ablation, and the neural challenger while retaining all leakage controls.

The default is `standard`. The locked benchmark is reserved for notebook `04`.


## Why dimensionality is controlled rather than simply discarded

The feature store deliberately contains many related estimates: different possession formulas, robust aggregations, rating systems, recency windows, and matchup transformations. Those alternatives are useful scientific hypotheses, but fitting all of them simultaneously to only a few thousand historical tournament games would create unstable estimates, redundant splits, inflated tuning variance, and unnecessary memory use.

This notebook therefore applies **training-fold-only block-aware stability selection**:

1. remove features with excessive missingness or near-zero variance;
2. measure univariate association separately by historical season;
3. reward stable direction and repeated availability across seasons;
4. enforce quotas so one large feature family cannot crowd out every other family;
5. retain pre-registered core basketball signals;
6. prune near-duplicate features by training-fold correlation;
7. cap the final dimension as a function of both the model family and the available training rows.

The selector is re-fit independently inside every inner and outer training fold. It never sees the corresponding validation outcomes.


In [ ]:
from __future__ import annotations

# Thread limits are set before importing numerical modeling libraries.
import os

_DEFAULT_THREAD_CAP = 4
for _variable in (
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
):
    os.environ.setdefault(_variable, str(_DEFAULT_THREAD_CAP))

import gc
import hashlib
import json
import math
import platform
import random
import re
import shutil
import sys
import time
import traceback
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import psutil
import scipy
import shap
import sklearn
import xgboost as xgb
import yaml
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit, logit
from scipy.stats import ks_2samp
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler
from threadpoolctl import threadpool_limits

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except Exception:
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None

try:
    from march_mania.paths import get_project_paths
except Exception as exc:
    raise ImportError(
        "The local march_mania package is unavailable. From the repository root, run "
        "`python -m pip install -e . --no-deps`, restart the kernel, and rerun."
    ) from exc

warnings.filterwarnings("once", category=RuntimeWarning)
# The environment can load both Intel and LLVM OpenMP runtimes; training is
# intentionally sequential and thread-capped. Suppress the repeated advisory
# while preserving all other RuntimeWarnings.
warnings.filterwarnings(
    "ignore",
    message=r"(?s).*Found Intel OpenMP.*LLVM OpenMP.*",
    category=RuntimeWarning,
)
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

RUN_MODE = os.environ.get("MM_RUN_MODE", "standard").strip().lower()
assert RUN_MODE in {"smoke", "standard", "exhaustive"}

PHYSICAL_CORES = psutil.cpu_count(logical=False) or 2
LOGICAL_CORES = psutil.cpu_count(logical=True) or PHYSICAL_CORES
MAX_THREADS = max(1, min(_DEFAULT_THREAD_CAP, PHYSICAL_CORES))
PROCESS = psutil.Process(os.getpid())
RAM_GB = psutil.virtual_memory().total / 1024**3

MODE_SETTINGS: dict[str, dict[str, Any]] = {
    "smoke": {
        "outer_folds_per_context": 1,
        "inner_fold_cap": 2,
        "linear_feature_cap": 40,
        "tree_feature_cap": 70,
        "margin_feature_cap": 70,
        "precorrelation_pool_cap": 120,
        "trial_budgets": {
            "elastic_logistic": 1,
            "xgb_classifier": 1,
            "lgb_classifier": 1,
            "xgb_margin": 1,
            "lgb_margin": 1,
        },
        "run_hist_classifier": True,
        "run_lgb_margin": False,
        "run_neural": False,
        "run_ablation": False,
        "run_explainability": False,
        "bootstrap_repetitions": 250,
        "shap_rows": 100,
    },
    "standard": {
        "outer_folds_per_context": None,
        "inner_fold_cap": 4,
        "linear_feature_cap": 80,
        "tree_feature_cap": 160,
        "margin_feature_cap": 140,
        "precorrelation_pool_cap": 300,
        "trial_budgets": {
            "elastic_logistic": 4,
            "xgb_classifier": 5,
            "lgb_classifier": 5,
            "xgb_margin": 5,
            "lgb_margin": 3,
        },
        "run_hist_classifier": True,
        "run_lgb_margin": True,
        "run_neural": False,
        "run_ablation": True,
        "run_explainability": True,
        "bootstrap_repetitions": 1000,
        "shap_rows": 400,
    },
    "exhaustive": {
        "outer_folds_per_context": None,
        "inner_fold_cap": 4,
        "linear_feature_cap": 110,
        "tree_feature_cap": 220,
        "margin_feature_cap": 190,
        "precorrelation_pool_cap": 420,
        "trial_budgets": {
            "elastic_logistic": 12,
            "xgb_classifier": 16,
            "lgb_classifier": 16,
            "xgb_margin": 14,
            "lgb_margin": 14,
            "torch_mlp": 8,
        },
        "run_hist_classifier": True,
        "run_lgb_margin": True,
        "run_neural": True,
        "run_ablation": True,
        "run_explainability": True,
        "bootstrap_repetitions": 3000,
        "shap_rows": 750,
    },
}
MODE = MODE_SETTINGS[RUN_MODE]

PATHS = get_project_paths()
ROOT = PATHS.root
INTERIM = PATHS.interim
PROCESSED = PATHS.processed
CONFIG_DIR = ROOT / "configs"
MODEL_REPORTS = ROOT / "reports" / "modeling" / "03_nested_oof"
FIGURE_DIR = ROOT / "reports" / "figures" / "modeling_03"
CACHE_DIR = ROOT / "data" / "model_cache" / "03_nested_oof" / RUN_MODE
STUDY_DIR = CACHE_DIR / "optuna"
FAILURE_DIR = CACHE_DIR / "failures"

for _directory in (
    CONFIG_DIR,
    MODEL_REPORTS,
    FIGURE_DIR,
    CACHE_DIR,
    STUDY_DIR,
    FAILURE_DIR,
):
    _directory.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Run mode:", RUN_MODE)
print("Python:", sys.executable)
print("Python version:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("LightGBM:", lgb.__version__)
print("Optuna:", optuna.__version__)
print("SHAP:", shap.__version__)
print("Physical/logical cores:", PHYSICAL_CORES, "/", LOGICAL_CORES)
print("Model threads:", MAX_THREADS)
print("System RAM GB:", round(RAM_GB, 2))
print("Current process RSS MB:", round(PROCESS.memory_info().rss / 1024**2, 2))

assert "ml-modeling" in str(sys.executable).lower(), (
    "Select the Python (ml-modeling) kernel before continuing."
)
if RAM_GB < 8:
    warnings.warn(
        "Less than 8 GB of system RAM detected. Use MM_RUN_MODE=smoke first; "
        "the notebook still projects columns and trains sequentially."
    )


## 1. Load and fingerprint the completed feature-store stage

Only the **historical** matchup store is loaded here. The much larger Stage 2 all-matchup matrix remains on disk until notebook `04`, after the development recipe is frozen.

The loader:

- verifies notebook `02` completed with zero blocking or symmetry failures;
- verifies the split and feature contract fingerprints;
- reads candidate manifests and feature provenance;
- projects only required columns from Parquet;
- reads only rows labeled `development`;
- never loads locked 2022–2025 outcomes into the model-selection frame.


In [ ]:
def sha256_file(path: Path, chunk_bytes: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def rss_mb() -> float:
    return PROCESS.memory_info().rss / 1024**2


split_path = CONFIG_DIR / "splits.yaml"
feature_path = CONFIG_DIR / "features.yaml"
readiness_01_path = ROOT / "reports" / "modeling" / "01_readiness_summary.json"
readiness_02_path = ROOT / "reports" / "feature_engineering" / "02_readiness_summary.json"
feature_checks_path = (
    ROOT
    / "reports"
    / "feature_engineering"
    / "feature_store_leakage_and_integrity_checks.csv"
)
candidate_sets_path = (
    ROOT / "reports" / "feature_engineering" / "candidate_feature_sets.json"
)
feature_registry_path = (
    ROOT / "reports" / "feature_engineering" / "feature_registry_v1.csv"
)
if not feature_registry_path.exists():
    feature_registry_path = (
        ROOT / "reports" / "feature_engineering" / "feature_registry.csv"
    )
artifact_manifest_path = (
    ROOT / "reports" / "feature_engineering" / "feature_store_artifact_manifest.csv"
)

required_control_files = [
    split_path,
    feature_path,
    readiness_01_path,
    readiness_02_path,
    feature_checks_path,
    candidate_sets_path,
    feature_registry_path,
    artifact_manifest_path,
]
missing_control = [str(path) for path in required_control_files if not path.exists()]
assert not missing_control, (
    "Required notebook 01/02 control artifacts are missing: "
    f"{missing_control}"
)

SPLITS = yaml.safe_load(split_path.read_text(encoding="utf-8"))
FEATURE_CONFIG = yaml.safe_load(feature_path.read_text(encoding="utf-8"))
READINESS_01 = read_json(readiness_01_path)
READINESS_02 = read_json(readiness_02_path)
FEATURE_CHECKS_02 = pd.read_csv(feature_checks_path)
CANDIDATE_SETS: dict[str, list[str]] = read_json(candidate_sets_path)
FEATURE_REGISTRY = pd.read_csv(feature_registry_path)
ARTIFACT_MANIFEST_02 = pd.read_csv(artifact_manifest_path)

assert READINESS_01["status"] == "complete"
assert READINESS_02["status"] == "complete"
assert READINESS_02["symmetry_failures"] == 0
assert READINESS_02["blocking_feature_check_failures"] == 0
assert FEATURE_CHECKS_02["Passed"].astype(bool).all()
assert READINESS_02["split_contract_sha256"] == SPLITS["contract_sha256"]
assert (
    READINESS_02["feature_contract_sha256"]
    == FEATURE_CONFIG["feature_contract_sha256"]
)
assert (
    FEATURE_CONFIG["source_split_contract_sha256"]
    == SPLITS["contract_sha256"]
)

HISTORICAL_STORE_PATH = (
    PROCESSED / "historical_matchup_feature_store_v1.parquet"
)
STAGE2_STORE_PATH = PROCESSED / "stage2_matchup_feature_store_v1.parquet"
OUTER_FOLDS_PATH = INTERIM / "fold_manifest_outer.parquet"
INNER_FOLDS_PATH = INTERIM / "fold_manifest_inner.parquet"
LOCKED_FOLDS_PATH = INTERIM / "fold_manifest_locked_benchmark.parquet"

required_data_files = [
    HISTORICAL_STORE_PATH,
    STAGE2_STORE_PATH,
    OUTER_FOLDS_PATH,
    INNER_FOLDS_PATH,
    LOCKED_FOLDS_PATH,
]
missing_data = [str(path) for path in required_data_files if not path.exists()]
assert not missing_data, f"Required data artifacts are missing: {missing_data}"

# The historical store is small enough to hash; Stage 2 is intentionally not
# read or re-hashed here because it is not part of model selection.
expected_historical_hash = READINESS_02["artifact_sha256"][
    "historical_matchup_feature_store_v1"
]
actual_historical_hash = sha256_file(HISTORICAL_STORE_PATH)
assert actual_historical_hash == expected_historical_hash, (
    "The historical feature store changed after notebook 02."
)

candidate_union = sorted(
    set().union(*(set(columns) for columns in CANDIDATE_SETS.values()))
)
assert len(candidate_union) == int(READINESS_02["candidate_feature_union"])

METADATA_COLUMNS = [
    "TargetKey",
    "GameKey",
    "Gender",
    "Season",
    "DayNum",
    "Team1ID",
    "Team2ID",
    "Team1Win",
    "Team1Margin",
    "DatasetRole",
    "CompactUniverseEligible",
    "RichUniverseEligible",
    "PrimaryModelRoute",
    "PooledChallengerEligible",
    "FeatureRoute",
]

# Read the Parquet schema without materializing the large matrix.
try:
    import pyarrow.parquet as pq

    historical_schema_columns = set(
        pq.ParquetFile(HISTORICAL_STORE_PATH).schema.names
    )
except Exception:
    # pandas can still read the file, but notebook 00/02 normally installed pyarrow.
    historical_schema_columns = set(
        pd.read_parquet(HISTORICAL_STORE_PATH).columns
    )

projected_columns = [
    column
    for column in METADATA_COLUMNS + candidate_union
    if column in historical_schema_columns
]
missing_metadata = {
    "TargetKey",
    "Gender",
    "Season",
    "Team1ID",
    "Team2ID",
    "Team1Win",
    "Team1Margin",
    "DatasetRole",
}.difference(projected_columns)
assert not missing_metadata, f"Historical store missing metadata: {missing_metadata}"

# The filter ensures locked labels never enter the development DataFrame.
development = pd.read_parquet(
    HISTORICAL_STORE_PATH,
    columns=projected_columns,
    filters=[("DatasetRole", "==", "development")],
)
outer_folds = pd.read_parquet(OUTER_FOLDS_PATH)
inner_folds = pd.read_parquet(INNER_FOLDS_PATH)
locked_fold_manifest = pd.read_parquet(LOCKED_FOLDS_PATH)

for column in development.select_dtypes(include=["float64"]).columns:
    development[column] = development[column].astype("float32")
for column in development.select_dtypes(include=["int64"]).columns:
    if column not in {"Season", "Team1ID", "Team2ID"}:
        minimum = development[column].min()
        maximum = development[column].max()
        if pd.notna(minimum) and pd.notna(maximum):
            if np.iinfo(np.int16).min <= minimum <= maximum <= np.iinfo(np.int16).max:
                development[column] = development[column].astype("int16")
            elif np.iinfo(np.int32).min <= minimum <= maximum <= np.iinfo(np.int32).max:
                development[column] = development[column].astype("int32")

LOCKED_SEASONS = set(map(int, SPLITS["locked_benchmark_seasons"]))
DEVELOPMENT_LAST_SEASON = int(SPLITS["development_last_season"])
TARGET_SEASON = int(SPLITS["target_season"])

assert development["DatasetRole"].eq("development").all()
assert development["Season"].max() <= DEVELOPMENT_LAST_SEASON
assert not development["Season"].isin(LOCKED_SEASONS).any()
assert development["TargetKey"].is_unique
assert development["Team1Win"].isin([0, 1]).all()
assert development["Team1ID"].lt(development["Team2ID"]).all()
numeric_development = development[candidate_union].select_dtypes(
    include=[np.number, "bool"]
)
numeric_development_array = numeric_development.to_numpy(
    dtype=np.float32,
    na_value=np.nan,
)
assert not np.isinf(numeric_development_array).any(), (
    "The development feature matrix contains positive or negative infinity. "
    "NaN is allowed for fold-fitted imputation, but infinity is not."
)
del numeric_development_array

print(
    json.dumps(
        {
            "development_rows_loaded": int(len(development)),
            "development_seasons": sorted(
                map(int, development["Season"].unique())
            ),
            "candidate_union": len(candidate_union),
            "historical_store_size_mb": round(
                HISTORICAL_STORE_PATH.stat().st_size / 1024**2, 3
            ),
            "stage2_store_size_mb_not_loaded": round(
                STAGE2_STORE_PATH.stat().st_size / 1024**2, 3
            ),
            "outer_folds": int(len(outer_folds)),
            "inner_folds": int(len(inner_folds)),
            "locked_manifest_rows_loaded_without_labels": int(
                len(locked_fold_manifest)
            ),
            "rss_mb_after_projected_load": round(rss_mb(), 2),
        },
        indent=2,
    )
)


## 2. Freeze the modeling and resource contract

This contract records the model families, feature caps, calibration methods, search rules, and ensemble constraints before development results are produced.

The scientific contract is independent of `RUN_MODE`. Runtime modes change the amount of computation, not the locked seasons, target definitions, temporal ordering, or legal feature universes.


In [ ]:
MODEL_CONFIG: dict[str, Any] = {
    "model_contract_version": 1,
    "source_split_contract_sha256": SPLITS["contract_sha256"],
    "source_feature_contract_sha256": FEATURE_CONFIG[
        "feature_contract_sha256"
    ],
    "target_season": TARGET_SEASON,
    "development_last_season": DEVELOPMENT_LAST_SEASON,
    "locked_benchmark_seasons": sorted(LOCKED_SEASONS),
    "primary_metric": "macro_mean_season_brier",
    "secondary_metrics": [
        "game_weighted_brier",
        "log_loss",
        "roc_auc",
        "calibration_intercept",
        "calibration_slope",
        "expected_calibration_error",
    ],
    "architectures": {
        "primary": ["separate_men", "separate_women"],
        "challengers": ["pooled_common", "partial_pooling"],
    },
    "targets": {
        "classification": "Team1Win",
        "margin_regression": "Team1Margin",
    },
    "feature_selection": {
        "fit_scope": "inner_or_outer_training_rows_only",
        "missingness_max": 0.45,
        "minimum_nonmissing_rows": 40,
        "near_zero_variance_tolerance": 1e-10,
        "correlation_prune_threshold": 0.985,
        "linear_feature_cap": MODE_SETTINGS["standard"][
            "linear_feature_cap"
        ],
        "tree_feature_cap": MODE_SETTINGS["standard"][
            "tree_feature_cap"
        ],
        "margin_feature_cap": MODE_SETTINGS["standard"][
            "margin_feature_cap"
        ],
        "row_adaptive_linear_divisor": 12,
        "row_adaptive_tree_divisor": 6,
        "block_balancing": True,
        "mandatory_core_signals": [
            "seed difference",
            "margin-aware Elo",
            "robust scoring margin",
            "schedule strength",
            "adjusted net efficiency",
            "Massey consensus when legally available",
            "gender context for pooled models",
        ],
    },
    "model_families": [
        "constant_probability",
        "seed_logistic",
        "elo_seed_logistic",
        "elastic_net_logistic",
        "hist_gradient_boosting",
        "xgboost_classifier",
        "lightgbm_classifier",
        "ridge_margin",
        "xgboost_margin",
        "lightgbm_margin",
        "optional_torch_mlp",
    ],
    "boosting_rules": {
        "outer_validation_never_used_for_early_stopping": True,
        "outer_round_count": "median inner-fold best iteration",
        "xgboost_tree_method": "hist",
        "xgboost_device": "cpu",
        "lightgbm_deterministic": True,
        "lightgbm_force_col_wise": True,
        "maximum_threads": MAX_THREADS,
    },
    "calibration": {
        "fit_source": "inner_oof_predictions_only",
        "probability_methods": [
            "identity",
            "temperature",
            "platt",
            "beta",
            "spline",
            "isotonic",
        ],
        "margin_methods": [
            "raw_platt",
            "raw_spline",
            "raw_isotonic",
        ],
        "selection": "cross_fitted_by_inner_validation_season_one_se_rule",
    },
    "ensemble": {
        "weights": "nonnegative_simplex",
        "objective": "macro_season_brier_plus_l2_penalty",
        "maximum_members": 5,
        "redundancy_correlation_threshold": 0.995,
        "partial_pooling": "gender_specific_prequential_blend",
    },
    "benchmark_boundary": {
        "locked_labels_loaded_in_notebook_03": False,
        "locked_evaluation_begins_in_notebook_04": True,
    },
}

model_contract_payload = json.dumps(
    MODEL_CONFIG, sort_keys=True, separators=(",", ":")
)
MODEL_CONFIG["model_contract_sha256"] = hashlib.sha256(
    model_contract_payload.encode("utf-8")
).hexdigest()

model_config_path = CONFIG_DIR / "modeling.yaml"
if model_config_path.exists():
    existing = yaml.safe_load(model_config_path.read_text(encoding="utf-8"))
    existing_without_hash = dict(existing)
    existing_without_hash.pop("model_contract_sha256", None)
    proposed_without_hash = dict(MODEL_CONFIG)
    proposed_without_hash.pop("model_contract_sha256", None)
    assert existing_without_hash == proposed_without_hash, (
        "configs/modeling.yaml already exists and differs from this "
        "notebook. Do not silently change the model-selection contract."
    )
else:
    model_config_path.write_text(
        yaml.safe_dump(MODEL_CONFIG, sort_keys=False),
        encoding="utf-8",
    )

(MODEL_REPORTS / "model_contract.json").write_text(
    json.dumps(MODEL_CONFIG, indent=2),
    encoding="utf-8",
)

print("Model contract:", model_config_path)
print("Model contract SHA-256:", MODEL_CONFIG["model_contract_sha256"])
print("Run-mode controls:", json.dumps(MODE, indent=2))


## 3. Atomic checkpoints, deterministic hashes, and memory guards

Every fold/model writes a self-contained checkpoint:

```text
data/model_cache/03_nested_oof/<OuterFoldID>/<ModelName>/
    metadata.json
    inner_oof.parquet
    outer_predictions.parquet
    selected_features.csv
    calibration_audit.csv
```

A checkpoint is reused only when the split, feature, and model-contract fingerprints match. This prevents stale predictions from being mixed into a new experiment.

Models are trained sequentially. The notebook never holds every fitted estimator in memory.


In [ ]:
def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        default=lambda x: (
            x.item()
            if isinstance(x, np.generic)
            else str(x)
        ),
    )


def object_sha256(value: Any) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def sanitize_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def atomic_write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, default=str),
        encoding="utf-8",
    )
    temporary.replace(path)


def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def atomic_write_parquet(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(
        temporary,
        index=False,
        compression="zstd",
    )
    temporary.replace(path)


def parse_json_int_list(value: str | float | None) -> list[int]:
    if value is None or (
        isinstance(value, float) and np.isnan(value)
    ):
        return []
    return [int(item) for item in json.loads(value)]


def clip_probability(probability: np.ndarray | pd.Series) -> np.ndarray:
    return np.clip(
        np.asarray(probability, dtype=np.float64),
        1e-6,
        1.0 - 1e-6,
    )


def release_memory(*objects: Any) -> None:
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()


def ensure_memory_headroom(
    stage: str,
    *,
    minimum_available_gb: float = 1.25,
    minimum_available_fraction: float = 0.06,
) -> None:
    """Fail cleanly before Windows is forced into an out-of-memory kill."""
    gc.collect()
    memory = psutil.virtual_memory()
    available_gb = memory.available / 1024**3
    available_fraction = memory.available / max(memory.total, 1)
    if (
        available_gb < minimum_available_gb
        or available_fraction < minimum_available_fraction
    ):
        raise MemoryError(
            f"Insufficient RAM headroom before {stage}: "
            f"{available_gb:.2f} GB available "
            f"({available_fraction:.1%}). Close other applications or "
            "resume in smoke mode. Completed checkpoints are preserved."
        )


RESOURCE_LOG: list[dict[str, Any]] = []


class ResourceTimer:
    def __init__(
        self,
        stage: str,
        *,
        outer_fold_id: str | None = None,
        model_name: str | None = None,
    ) -> None:
        self.stage = stage
        self.outer_fold_id = outer_fold_id
        self.model_name = model_name

    def __enter__(self) -> "ResourceTimer":
        self.start_time = time.perf_counter()
        self.start_rss = rss_mb()
        self.start_available_gb = (
            psutil.virtual_memory().available / 1024**3
        )
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        end_rss = rss_mb()
        end_available_gb = psutil.virtual_memory().available / 1024**3
        RESOURCE_LOG.append(
            {
                "Stage": self.stage,
                "OuterFoldID": self.outer_fold_id,
                "Model": self.model_name,
                "ElapsedSeconds": time.perf_counter() - self.start_time,
                "StartRSSMB": self.start_rss,
                "EndRSSMB": end_rss,
                "RSSDeltaMB": end_rss - self.start_rss,
                "AvailableRAMGBStart": self.start_available_gb,
                "AvailableRAMGBEnd": end_available_gb,
                "Succeeded": exc_type is None,
            }
        )


EXPECTED_CONTRACTS = {
    "split": SPLITS["contract_sha256"],
    "feature": FEATURE_CONFIG["feature_contract_sha256"],
    "model": MODEL_CONFIG["model_contract_sha256"],
    "historical_store": expected_historical_hash,
}


def checkpoint_is_valid(directory: Path) -> bool:
    metadata_path = directory / "metadata.json"
    prediction_path = directory / "outer_predictions.parquet"
    inner_path = directory / "inner_oof.parquet"
    if not (
        metadata_path.exists()
        and prediction_path.exists()
        and inner_path.exists()
    ):
        return False
    try:
        metadata = read_json(metadata_path)
    except Exception:
        return False
    return (
        metadata.get("status") == "complete"
        and metadata.get("contracts") == EXPECTED_CONTRACTS
    )


print("Checkpoint root:", CACHE_DIR)
print("Expected contracts:", json.dumps(EXPECTED_CONTRACTS, indent=2))


## 4. Feature provenance and block-aware stability selection

The broad feature bank is never fed directly to a model. The selector below is deliberately model-family-aware:

- **linear models** receive the smallest, most stable subset;
- **tree models** receive a larger but still bounded subset;
- **margin models** use their own target-specific rankings;
- **baseline models** use transparent pre-registered columns only.

Association is estimated across training seasons, not by one pooled correlation alone. Features are rewarded for:

- repeated availability;
- stable sign;
- median absolute within-season association;
- global training association;
- low missingness.

The final correlation-pruning step operates only on a pre-screened pool, keeping its memory cost bounded.


In [ ]:
FEATURE_REGISTRY = FEATURE_REGISTRY.drop_duplicates("Feature").copy()
REGISTRY_BY_FEATURE = (
    FEATURE_REGISTRY.set_index("Feature").to_dict(orient="index")
)


def underlying_feature_name(column: str) -> str:
    for prefix in ("diff__", "absdiff__", "mean__", "max__", "min__"):
        if column.startswith(prefix):
            return column[len(prefix):]
    return column


def infer_feature_block(column: str) -> str:
    if column in REGISTRY_BY_FEATURE:
        return str(REGISTRY_BY_FEATURE[column].get("Block", "unregistered"))
    base = underlying_feature_name(column)
    if base in REGISTRY_BY_FEATURE:
        return str(REGISTRY_BY_FEATURE[base].get("Block", "unregistered"))
    if column.startswith("seedprior__"):
        return "prequential_seed_priors"
    if column.startswith("matchup__seed"):
        return "selection_committee_prior"
    if column.startswith("interaction__"):
        return "matchup_interactions"
    if column.startswith("context__") or column.startswith("availability__"):
        return "availability_and_context"
    if "massey__" in column:
        return "massey_consensus"
    if "coach__" in column:
        return "prior_coach_history"
    if "prior__" in column:
        return "prior_program_history"
    if "schedule__" in column:
        return "schedule_and_accomplishment"
    if "conference__" in column:
        return "conference_context"
    if "adj__" in column:
        return "opponent_adjusted_efficiency"
    if "rating__" in column:
        return "dynamic_ratings"
    if "detailed__" in column:
        return "detailed_efficiency"
    if "compact__" in column:
        return "compact_performance"
    if "norm__" in column:
        return "within_season_normalization"
    return "other"


FEATURE_TO_BLOCK = {
    column: infer_feature_block(column)
    for column in candidate_union
}

BLOCK_BASE_QUOTAS = {
    "selection_committee_prior": 14,
    "prequential_seed_priors": 10,
    "compact_performance": 30,
    "dynamic_ratings": 24,
    "global_strength_ratings": 20,
    "schedule_and_accomplishment": 22,
    "conference_context": 10,
    "detailed_efficiency": 34,
    "opponent_adjusted_efficiency": 18,
    "prior_program_history": 10,
    "prior_coach_history": 7,
    "massey_consensus": 24,
    "matchup_interactions": 12,
    "within_season_normalization": 18,
    "availability_and_context": 10,
    "geography": 8,
    "other": 8,
}

MANDATORY_TOKEN_GROUPS = [
    ("matchup__seed_diff",),
    ("rating__elo_mov_538", "rating__elo_mov_log", "rating__elo_standard"),
    ("compact__margin_trim02", "compact__margin_mean"),
    ("schedule__opponent_strength_mean", "schedule__rpi"),
    ("adj__net_rtg_a50p0", "adj__net_rtg_a10p0"),
    ("massey__strength_mean", "massey__rank_mean"),
]


def first_matching_feature(
    candidates: Sequence[str],
    tokens: Sequence[str],
) -> str | None:
    for token in tokens:
        exact_or_containing = [
            column
            for column in candidates
            if column == token
            or column.endswith(token)
            or token in column
        ]
        directional = [
            column
            for column in exact_or_containing
            if column.startswith("diff__")
            or column.startswith("matchup__")
            or column.startswith("interaction__")
        ]
        if directional:
            return sorted(directional)[0]
        if exact_or_containing:
            return sorted(exact_or_containing)[0]
    return None


def mandatory_features(
    candidates: Sequence[str],
    *,
    pooled: bool,
) -> list[str]:
    chosen: list[str] = []
    for tokens in MANDATORY_TOKEN_GROUPS:
        match = first_matching_feature(candidates, tokens)
        if match is not None:
            chosen.append(match)
    if pooled and "context__women" in candidates:
        chosen.append("context__women")
    return list(dict.fromkeys(chosen))


def baseline_features(
    model_name: str,
    candidates: Sequence[str],
    *,
    pooled: bool,
) -> list[str]:
    candidate_set = set(candidates)
    if model_name == "constant_probability":
        return []

    if model_name == "seed_logistic":
        ordered = [
            column
            for column in candidates
            if (
                column == "matchup__seed_diff"
                or column == "matchup__seed_gap_abs"
                or column == "matchup__team1_is_seed_favorite"
                or column == "matchup__equal_seed"
                or column.startswith("seedprior__")
            )
        ]
        # Keep a transparent baseline rather than every redundant seed prior.
        preferred = [
            column
            for column in ordered
            if any(
                token in column
                for token in (
                    "matchup__seed_diff",
                    "matchup__seed_gap_abs",
                    "team1_is_seed_favorite",
                    "pc8",
                    "overall",
                )
            )
        ]
        selected = preferred[:12] if preferred else ordered[:12]
    elif model_name == "elo_seed_logistic":
        selected = baseline_features(
            "seed_logistic", candidates, pooled=pooled
        )
        elo = [
            column
            for column in candidates
            if (
                "rating__elo_mov_538" in column
                or "rating__elo_mov_log" in column
                or "rating__elo_standard" in column
            )
            and column.startswith(("diff__", "absdiff__"))
        ]
        selected += sorted(elo)[:12]
    else:
        selected = []

    if pooled and "context__women" in candidate_set:
        selected.append("context__women")
    return list(dict.fromkeys(selected))


def vectorized_correlations(
    frame: pd.DataFrame,
    features: Sequence[str],
    target: np.ndarray,
) -> np.ndarray:
    if not features:
        return np.array([], dtype=np.float64)
    # pandas 3 may expose a read-only NumPy view. The selector performs
    # fold-local in-place centering/imputation, so request an explicit writable copy.
    matrix = np.array(
        frame[list(features)].to_numpy(
            dtype=np.float64,
            na_value=np.nan,
        ),
        dtype=np.float64,
        copy=True,
        order="C",
    )
    medians = np.nanmedian(matrix, axis=0)
    medians = np.where(np.isfinite(medians), medians, 0.0)
    missing = ~np.isfinite(matrix)
    if missing.any():
        matrix[missing] = np.take(medians, np.where(missing)[1])
    matrix -= matrix.mean(axis=0, keepdims=True)
    centered_target = target.astype(np.float64) - np.mean(target)
    numerator = matrix.T @ centered_target
    denominator = np.sqrt(
        np.sum(matrix * matrix, axis=0)
        * np.sum(centered_target * centered_target)
    )
    correlation = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator, dtype=np.float64),
        where=denominator > 0,
    )
    return correlation


@dataclass
class SelectionResult:
    selected_features: list[str]
    audit: pd.DataFrame
    effective_cap: int
    precorrelation_pool: list[str]
    mandatory_features: list[str]
    selector_hash: str


def effective_feature_cap(
    *,
    family: str,
    training_rows: int,
) -> int:
    if family == "linear":
        fixed = int(MODE["linear_feature_cap"])
        adaptive = max(12, training_rows // 12)
    elif family == "margin":
        fixed = int(MODE["margin_feature_cap"])
        adaptive = max(20, training_rows // 7)
    else:
        fixed = int(MODE["tree_feature_cap"])
        adaptive = max(24, training_rows // 6)
    return max(8, min(fixed, adaptive))


def fit_block_aware_selector(
    training: pd.DataFrame,
    candidates: Sequence[str],
    *,
    target_column: str,
    family: str,
    pooled: bool,
) -> SelectionResult:
    assert target_column in {"Team1Win", "Team1Margin"}
    candidates = [
        column
        for column in candidates
        if column in training.columns
        and pd.api.types.is_numeric_dtype(training[column])
    ]
    assert candidates, "No candidate features are available."

    target = pd.to_numeric(
        training[target_column], errors="coerce"
    ).to_numpy(dtype=np.float64)
    assert np.isfinite(target).all()

    missingness = training[candidates].isna().mean()
    nonmissing = training[candidates].notna().sum()
    unique_counts = training[candidates].nunique(dropna=True)

    eligible = [
        column
        for column in candidates
        if missingness[column]
        <= float(MODEL_CONFIG["feature_selection"]["missingness_max"])
        and nonmissing[column]
        >= min(
            int(MODEL_CONFIG["feature_selection"]["minimum_nonmissing_rows"]),
            len(training),
        )
        and unique_counts[column] > 1
    ]

    mandatory = [
        column
        for column in mandatory_features(candidates, pooled=pooled)
        if column in candidates
        and unique_counts.get(column, 0) > 1
    ]
    eligible = list(dict.fromkeys(mandatory + eligible))
    assert eligible, "Every feature was removed by eligibility filters."

    global_corr = vectorized_correlations(training, eligible, target)
    per_group_corr: dict[str, list[float]] = {
        feature: [] for feature in eligible
    }

    group_columns = ["Season", "Gender"] if pooled else ["Season"]
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for _, group in training.groupby(grouper, observed=True, sort=True):
        if len(group) < 20:
            continue
        group_target = pd.to_numeric(
            group[target_column], errors="coerce"
        ).to_numpy(dtype=np.float64)
        if np.nanstd(group_target) <= 0:
            continue
        correlations = vectorized_correlations(
            group, eligible, group_target
        )
        for feature, value in zip(eligible, correlations, strict=True):
            if np.isfinite(value):
                per_group_corr[feature].append(float(value))

    records: list[dict[str, Any]] = []
    for index, feature in enumerate(eligible):
        correlations = np.asarray(
            per_group_corr[feature], dtype=np.float64
        )
        nonzero = correlations[np.abs(correlations) > 1e-12]
        sign_consistency = (
            abs(np.mean(np.sign(nonzero)))
            if nonzero.size
            else 0.0
        )
        group_coverage = (
            len(correlations)
            / max(
                1,
                training[group_columns]
                .drop_duplicates()
                .shape[0],
            )
        )
        median_abs = (
            float(np.median(np.abs(correlations)))
            if correlations.size
            else 0.0
        )
        stability_iqr = (
            float(
                np.subtract(
                    *np.percentile(correlations, [75, 25])
                )
            )
            if correlations.size >= 2
            else 0.0
        )
        score = (
            0.48 * median_abs
            + 0.22 * abs(float(global_corr[index]))
            + 0.16 * sign_consistency
            + 0.10 * math.sqrt(max(group_coverage, 0.0))
            - 0.08 * float(missingness[feature])
            - 0.04 * min(stability_iqr, 1.0)
        )
        records.append(
            {
                "Feature": feature,
                "Block": FEATURE_TO_BLOCK.get(feature, "other"),
                "MissingRate": float(missingness[feature]),
                "NonMissingRows": int(nonmissing[feature]),
                "UniqueValues": int(unique_counts[feature]),
                "GlobalCorrelation": float(global_corr[index]),
                "MedianAbsoluteSeasonCorrelation": median_abs,
                "SeasonSignConsistency": float(sign_consistency),
                "SeasonCoverage": float(group_coverage),
                "SeasonCorrelationIQR": stability_iqr,
                "StabilityScore": float(score),
                "Mandatory": feature in mandatory,
            }
        )

    audit = pd.DataFrame(records).sort_values(
        ["Mandatory", "StabilityScore", "Feature"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    cap = effective_feature_cap(
        family=family,
        training_rows=len(training),
    )
    pool_cap = min(
        int(MODE["precorrelation_pool_cap"]),
        max(cap * 2, cap + 20),
    )

    # Scale block quotas to the requested family cap.
    quota_scale = cap / max(1, sum(BLOCK_BASE_QUOTAS.values()))
    block_choices: list[str] = []
    for block, group in audit.groupby("Block", observed=True, sort=False):
        base_quota = BLOCK_BASE_QUOTAS.get(block, 8)
        quota = max(
            2,
            min(
                base_quota,
                int(math.ceil(base_quota * max(0.65, quota_scale * 5))),
            ),
        )
        block_choices.extend(group.head(quota)["Feature"].tolist())

    ranked = audit["Feature"].tolist()
    pool = list(dict.fromkeys(mandatory + block_choices + ranked))
    pool = pool[:pool_cap]

    # Correlation pruning is bounded to the pre-screened pool.
    # Explicit writable copy is required because correlation pruning performs
    # in-place median imputation and pandas 3 can return a read-only array.
    pool_matrix = np.array(
        training[pool].to_numpy(
            dtype=np.float32,
            na_value=np.nan,
        ),
        dtype=np.float32,
        copy=True,
        order="C",
    )
    medians = np.nanmedian(pool_matrix, axis=0)
    medians = np.where(np.isfinite(medians), medians, 0.0).astype(
        np.float32
    )
    missing = ~np.isfinite(pool_matrix)
    if missing.any():
        pool_matrix[missing] = np.take(
            medians, np.where(missing)[1]
        )
    standard_deviation = pool_matrix.std(axis=0)
    safe_std = np.where(standard_deviation > 1e-12, standard_deviation, 1.0)
    normalized = (
        pool_matrix - pool_matrix.mean(axis=0, keepdims=True)
    ) / safe_std
    correlation = np.abs(
        np.clip(
            (normalized.T @ normalized)
            / max(1, normalized.shape[0] - 1),
            -1,
            1,
        )
    )

    selected: list[str] = []
    selected_indices: list[int] = []
    threshold = float(
        MODEL_CONFIG["feature_selection"]["correlation_prune_threshold"]
    )
    for index, feature in enumerate(pool):
        if feature in mandatory:
            selected.append(feature)
            selected_indices.append(index)
            continue
        if selected_indices and np.any(
            correlation[index, selected_indices] >= threshold
        ):
            continue
        selected.append(feature)
        selected_indices.append(index)
        if len(selected) >= cap:
            break

    # Mandatory features survive even if mutually correlated; trim only
    # nonmandatory tail if needed.
    if len(selected) > cap:
        mandatory_set = set(mandatory)
        retained = [f for f in selected if f in mandatory_set]
        retained += [
            f for f in selected if f not in mandatory_set
        ][: max(0, cap - len(retained))]
        selected = retained

    selected_set = set(selected)
    pool_set = set(pool)
    audit["InPrecorrelationPool"] = audit["Feature"].isin(pool_set)
    audit["Selected"] = audit["Feature"].isin(selected_set)
    audit["SelectionRank"] = audit["Feature"].map(
        {feature: index + 1 for index, feature in enumerate(selected)}
    )
    audit["ExclusionReason"] = np.select(
        [
            audit["Selected"],
            ~audit["InPrecorrelationPool"],
        ],
        [
            "selected",
            "below_bounded_prescreen",
        ],
        default="correlation_pruned_or_cap",
    )

    selector_payload = {
        "target": target_column,
        "family": family,
        "pooled": pooled,
        "training_seasons": sorted(
            map(int, training["Season"].unique())
        ),
        "training_rows": len(training),
        "candidate_hash": object_sha256(sorted(candidates)),
        "selected": selected,
        "contracts": EXPECTED_CONTRACTS,
    }
    selector_hash = object_sha256(selector_payload)
    return SelectionResult(
        selected_features=selected,
        audit=audit,
        effective_cap=cap,
        precorrelation_pool=pool,
        mandatory_features=mandatory,
        selector_hash=selector_hash,
    )


SELECTION_MEMO: dict[str, SelectionResult] = {}


def get_selection(
    training: pd.DataFrame,
    candidates: Sequence[str],
    *,
    target_column: str,
    family: str,
    pooled: bool,
    selection_key: str,
) -> SelectionResult:
    payload = {
        "selection_key": selection_key,
        "target_column": target_column,
        "family": family,
        "pooled": pooled,
        "training_seasons": sorted(
            map(int, training["Season"].unique())
        ),
        "training_rows": len(training),
        "candidate_hash": object_sha256(sorted(candidates)),
        "mode_caps": {
            "linear": MODE["linear_feature_cap"],
            "tree": MODE["tree_feature_cap"],
            "margin": MODE["margin_feature_cap"],
            "pool": MODE["precorrelation_pool_cap"],
        },
        "contracts": EXPECTED_CONTRACTS,
    }
    key = object_sha256(payload)
    if key in SELECTION_MEMO:
        return SELECTION_MEMO[key]

    directory = CACHE_DIR / "selectors" / key
    metadata_path = directory / "metadata.json"
    audit_path = directory / "audit.csv"
    if metadata_path.exists() and audit_path.exists():
        metadata = read_json(metadata_path)
        if metadata.get("payload") == payload:
            result = SelectionResult(
                selected_features=list(metadata["selected_features"]),
                audit=pd.read_csv(audit_path),
                effective_cap=int(metadata["effective_cap"]),
                precorrelation_pool=list(metadata["precorrelation_pool"]),
                mandatory_features=list(metadata["mandatory_features"]),
                selector_hash=str(metadata["selector_hash"]),
            )
            SELECTION_MEMO[key] = result
            return result

    result = fit_block_aware_selector(
        training,
        candidates,
        target_column=target_column,
        family=family,
        pooled=pooled,
    )
    directory.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(audit_path, result.audit)
    atomic_write_json(
        metadata_path,
        {
            "payload": payload,
            "selected_features": result.selected_features,
            "effective_cap": result.effective_cap,
            "precorrelation_pool": result.precorrelation_pool,
            "mandatory_features": result.mandatory_features,
            "selector_hash": result.selector_hash,
        },
    )
    SELECTION_MEMO[key] = result
    return result


block_inventory = (
    pd.Series(FEATURE_TO_BLOCK, name="Block")
    .rename_axis("Feature")
    .reset_index()
    .groupby("Block", observed=True)
    .size()
    .sort_values(ascending=False)
    .to_frame("CandidateFeatures")
)
block_inventory.to_csv(
    MODEL_REPORTS / "candidate_feature_blocks.csv"
)
block_inventory


### Selector preflight

Before fitting thousands of bounded models, run the selector once on the latest legal rich men’s and women’s development histories. This verifies:

- the broad candidate sets can be resolved;
- dimensions are reduced sharply;
- mandatory core signals survive when available;
- no selected set exceeds its adaptive cap.


In [ ]:
def candidate_set_name(
    *,
    gender: str,
    universe: str,
    seed_aware: bool = True,
) -> str:
    route = "seed_aware" if seed_aware else "seed_free"
    if gender == "Pooled":
        assert universe == "rich"
        return f"pooled_rich_{route}"
    prefix = "men" if gender == "M" else "women"
    return f"{prefix}_{universe}_{route}"


def context_mask(
    frame: pd.DataFrame,
    *,
    seasons: Sequence[int],
    gender: str,
    universe: str,
) -> pd.Series:
    mask = frame["Season"].isin(list(map(int, seasons)))
    if gender != "Pooled":
        mask &= frame["Gender"].eq(gender)
    eligibility_column = (
        "CompactUniverseEligible"
        if universe == "compact"
        else "RichUniverseEligible"
    )
    if eligibility_column in frame.columns:
        mask &= frame[eligibility_column].astype(bool)
    return mask


selector_preflight_records: list[dict[str, Any]] = []
for gender in ("M", "W", "Pooled"):
    universe = "rich"
    key = candidate_set_name(
        gender=gender,
        universe=universe,
        seed_aware=True,
    )
    candidates = [
        column
        for column in CANDIDATE_SETS[key]
        if column in development.columns
    ]
    seasons = sorted(
        map(
            int,
            development.loc[
                context_mask(
                    development,
                    seasons=development["Season"].unique(),
                    gender=gender,
                    universe=universe,
                ),
                "Season",
            ].unique(),
        )
    )
    training = development.loc[
        context_mask(
            development,
            seasons=seasons,
            gender=gender,
            universe=universe,
        )
    ].copy()
    result = fit_block_aware_selector(
        training,
        candidates,
        target_column="Team1Win",
        family="tree",
        pooled=gender == "Pooled",
    )
    assert len(result.selected_features) <= result.effective_cap
    selector_preflight_records.append(
        {
            "GenderContext": gender,
            "TrainingRows": len(training),
            "CandidateFeatures": len(candidates),
            "PrecorrelationPool": len(result.precorrelation_pool),
            "AdaptiveCap": result.effective_cap,
            "SelectedFeatures": len(result.selected_features),
            "SelectedBlocks": result.audit.loc[
                result.audit["Selected"], "Block"
            ].nunique(),
            "MandatoryRetained": len(
                set(result.mandatory_features)
                .intersection(result.selected_features)
            ),
        }
    )
    del training, result
    gc.collect()

selector_preflight = pd.DataFrame(selector_preflight_records)
selector_preflight.to_csv(
    MODEL_REPORTS / "selector_preflight.csv",
    index=False,
)
selector_preflight


## 5. Probability, margin, calibration, and uncertainty metrics

The primary score is the **unweighted mean of season-level Brier scores**. This prevents seasons with slightly more tournament games from dominating selection.

Every model also receives:

- game-weighted Brier score;
- log loss;
- ROC AUC;
- accuracy at 0.50;
- calibration intercept and slope;
- expected and maximum calibration error;
- Brier reliability, resolution, and uncertainty components;
- worst-season Brier and across-season standard deviation.

Uncertainty is estimated by resampling **whole seasons**, not individual games.


In [ ]:
def season_metric_values(
    frame: pd.DataFrame,
    probability_column: str,
    *,
    outcome_column: str = "Team1Win",
) -> pd.DataFrame:
    rows = []
    for season, group in frame.groupby("Season", observed=True, sort=True):
        y = group[outcome_column].to_numpy(dtype=int)
        p = clip_probability(group[probability_column])
        rows.append(
            {
                "Season": int(season),
                "Rows": len(group),
                "Brier": float(np.mean((p - y) ** 2)),
                "LogLoss": float(log_loss(y, p, labels=[0, 1])),
                "ROCAUC": (
                    float(roc_auc_score(y, p))
                    if np.unique(y).size == 2
                    else np.nan
                ),
                "Accuracy": float(
                    accuracy_score(y, (p >= 0.5).astype(int))
                ),
            }
        )
    return pd.DataFrame(rows)


def calibration_intercept_slope(
    y: np.ndarray,
    p: np.ndarray,
) -> tuple[float, float]:
    y = np.asarray(y, dtype=int)
    p = clip_probability(p)
    if np.unique(y).size < 2:
        return np.nan, np.nan
    x = logit(p).reshape(-1, 1)
    try:
        model = LogisticRegression(
            C=1e6,
            solver="lbfgs",
            max_iter=5000,
        )
        model.fit(x, y)
        return float(model.intercept_[0]), float(model.coef_[0, 0])
    except Exception:
        return np.nan, np.nan


def calibration_error(
    y: np.ndarray,
    p: np.ndarray,
    *,
    bins: int = 10,
) -> tuple[float, float]:
    y = np.asarray(y, dtype=float)
    p = clip_probability(p)
    if len(p) == 0:
        return np.nan, np.nan
    try:
        quantile_bins = pd.qcut(
            pd.Series(p),
            q=min(bins, max(2, len(np.unique(p)))),
            duplicates="drop",
        )
    except Exception:
        quantile_bins = pd.cut(
            pd.Series(p),
            bins=min(bins, max(2, len(np.unique(p)))),
            duplicates="drop",
        )
    table = pd.DataFrame({"y": y, "p": p, "bin": quantile_bins})
    grouped = table.groupby("bin", observed=True).agg(
        rows=("y", "size"),
        observed=("y", "mean"),
        predicted=("p", "mean"),
    )
    if grouped.empty:
        return np.nan, np.nan
    gap = np.abs(grouped["observed"] - grouped["predicted"])
    ece = float(np.average(gap, weights=grouped["rows"]))
    mce = float(gap.max())
    return ece, mce


def brier_decomposition(
    y: np.ndarray,
    p: np.ndarray,
    *,
    bins: int = 10,
) -> dict[str, float]:
    y = np.asarray(y, dtype=float)
    p = clip_probability(p)
    climatology = float(np.mean(y))
    uncertainty = climatology * (1.0 - climatology)
    try:
        labels = pd.qcut(
            pd.Series(p),
            q=min(bins, max(2, len(np.unique(p)))),
            duplicates="drop",
        )
    except Exception:
        labels = pd.cut(
            pd.Series(p),
            bins=min(bins, max(2, len(np.unique(p)))),
            duplicates="drop",
        )
    frame = pd.DataFrame({"y": y, "p": p, "bin": labels})
    grouped = frame.groupby("bin", observed=True).agg(
        rows=("y", "size"),
        observed=("y", "mean"),
        predicted=("p", "mean"),
    )
    if grouped.empty:
        return {
            "Reliability": np.nan,
            "Resolution": np.nan,
            "Uncertainty": uncertainty,
        }
    weights = grouped["rows"] / grouped["rows"].sum()
    reliability = float(
        np.sum(
            weights
            * (grouped["predicted"] - grouped["observed"]) ** 2
        )
    )
    resolution = float(
        np.sum(weights * (grouped["observed"] - climatology) ** 2)
    )
    return {
        "Reliability": reliability,
        "Resolution": resolution,
        "Uncertainty": uncertainty,
    }


def evaluate_probability_frame(
    frame: pd.DataFrame,
    probability_column: str = "Prediction",
) -> tuple[dict[str, Any], pd.DataFrame]:
    y = frame["Team1Win"].to_numpy(dtype=int)
    p = clip_probability(frame[probability_column])
    by_season = season_metric_values(frame, probability_column)
    intercept, slope = calibration_intercept_slope(y, p)
    ece, mce = calibration_error(y, p)
    decomposition = brier_decomposition(y, p)
    metrics = {
        "Rows": int(len(frame)),
        "Seasons": int(frame["Season"].nunique()),
        "MacroSeasonBrier": float(by_season["Brier"].mean()),
        "GameWeightedBrier": float(np.mean((p - y) ** 2)),
        "SeasonBrierStd": float(by_season["Brier"].std(ddof=1))
        if len(by_season) > 1
        else 0.0,
        "WorstSeasonBrier": float(by_season["Brier"].max()),
        "LogLoss": float(log_loss(y, p, labels=[0, 1])),
        "ROCAUC": (
            float(roc_auc_score(y, p))
            if np.unique(y).size == 2
            else np.nan
        ),
        "Accuracy": float(
            accuracy_score(y, (p >= 0.5).astype(int))
        ),
        "CalibrationIntercept": intercept,
        "CalibrationSlope": slope,
        "ECE": ece,
        "MCE": mce,
        **decomposition,
    }
    return metrics, by_season


def robust_probability_objective(frame: pd.DataFrame) -> float:
    metrics, _ = evaluate_probability_frame(frame)
    return float(
        metrics["MacroSeasonBrier"]
        + 0.10 * metrics["SeasonBrierStd"]
        + 0.05
        * max(
            0.0,
            metrics["WorstSeasonBrier"]
            - metrics["MacroSeasonBrier"],
        )
    )


def robust_margin_objective(
    frame: pd.DataFrame,
    prediction_column: str = "RawPrediction",
) -> float:
    season_rmse = []
    for _, group in frame.groupby("Season", observed=True, sort=True):
        error = (
            group[prediction_column].to_numpy(dtype=float)
            - group["Team1Margin"].to_numpy(dtype=float)
        )
        season_rmse.append(float(np.sqrt(np.mean(error**2))))
    if not season_rmse:
        return float("inf")
    values = np.asarray(season_rmse, dtype=float)
    return float(values.mean() + 0.10 * values.std(ddof=0))


def season_cluster_bootstrap_difference(
    frame: pd.DataFrame,
    *,
    probability_a: str,
    probability_b: str,
    repetitions: int,
    seed: int = SEED,
) -> dict[str, float]:
    seasons = np.array(
        sorted(map(int, frame["Season"].unique())),
        dtype=int,
    )
    if seasons.size < 2:
        return {
            "MeanBrierDifferenceAminusB": np.nan,
            "CILower": np.nan,
            "CIUpper": np.nan,
            "ProbabilityALowerBrier": np.nan,
        }
    rng = np.random.default_rng(seed)
    observed = []
    for season in seasons:
        group = frame.loc[frame["Season"].eq(season)]
        y = group["Team1Win"].to_numpy(dtype=float)
        loss_a = (
            clip_probability(group[probability_a]) - y
        ) ** 2
        loss_b = (
            clip_probability(group[probability_b]) - y
        ) ** 2
        observed.append(float(np.mean(loss_a - loss_b)))
    observed = np.asarray(observed, dtype=float)
    draws = np.empty(repetitions, dtype=float)
    for index in range(repetitions):
        sample = rng.choice(
            observed, size=len(observed), replace=True
        )
        draws[index] = sample.mean()
    return {
        "MeanBrierDifferenceAminusB": float(observed.mean()),
        "CILower": float(np.quantile(draws, 0.025)),
        "CIUpper": float(np.quantile(draws, 0.975)),
        "ProbabilityALowerBrier": float(np.mean(draws < 0)),
    }


print("Metric utilities ready.")


## 6. Cross-fitted calibration

Calibration is treated as a model component, not a cosmetic post-processing step.

For each outer fold:

1. the chosen base model generates inner out-of-fold raw predictions;
2. each candidate calibrator is evaluated by leaving out one inner validation season at a time;
3. the one-standard-error rule prefers the simplest method statistically indistinguishable from the best;
4. the selected calibrator is fit on all inner out-of-fold predictions;
5. it transforms the untouched outer-fold prediction.

Supported probability calibrators:

- identity;
- temperature scaling;
- Platt scaling;
- beta calibration;
- monotonic spline logistic calibration;
- isotonic regression.

Margin models use Platt, spline, or isotonic mappings from predicted point margin to win probability.


In [ ]:
class ConstantCalibrator:
    def __init__(self, probability: float) -> None:
        self.probability = float(probability)

    def predict(self, raw: np.ndarray) -> np.ndarray:
        return np.full(
            len(np.asarray(raw)),
            self.probability,
            dtype=np.float64,
        )


class IdentityCalibrator:
    def predict(self, raw: np.ndarray) -> np.ndarray:
        return clip_probability(raw)


class TemperatureCalibrator:
    def __init__(self, temperature: float) -> None:
        self.temperature = float(temperature)

    def predict(self, raw: np.ndarray) -> np.ndarray:
        logits = logit(clip_probability(raw))
        return clip_probability(expit(logits / self.temperature))


class SklearnCalibrator:
    def __init__(
        self,
        estimator: Any,
        transform: Callable[[np.ndarray], np.ndarray],
    ) -> None:
        self.estimator = estimator
        self.transform = transform

    def predict(self, raw: np.ndarray) -> np.ndarray:
        transformed = self.transform(np.asarray(raw, dtype=float))
        if hasattr(self.estimator, "predict_proba"):
            probability = self.estimator.predict_proba(
                transformed
            )[:, 1]
        else:
            probability = self.estimator.predict(transformed)
        return clip_probability(probability)


class IsotonicCalibrator:
    def __init__(self, estimator: IsotonicRegression) -> None:
        self.estimator = estimator

    def predict(self, raw: np.ndarray) -> np.ndarray:
        return clip_probability(
            self.estimator.predict(np.asarray(raw, dtype=float))
        )


def transform_probability_logit(raw: np.ndarray) -> np.ndarray:
    return logit(clip_probability(raw)).reshape(-1, 1)


def transform_beta(raw: np.ndarray) -> np.ndarray:
    p = clip_probability(raw)
    return np.column_stack([np.log(p), -np.log1p(-p)])


def transform_raw(raw: np.ndarray) -> np.ndarray:
    return np.asarray(raw, dtype=float).reshape(-1, 1)


def fit_calibrator(
    method: str,
    raw: np.ndarray,
    y: np.ndarray,
    *,
    raw_kind: str,
) -> Any:
    raw = np.asarray(raw, dtype=float)
    y = np.asarray(y, dtype=int)
    if len(raw) == 0:
        raise ValueError("Cannot fit a calibrator without rows.")
    if np.unique(y).size < 2:
        return ConstantCalibrator(float(np.mean(y)))

    if method == "identity":
        assert raw_kind == "probability"
        return IdentityCalibrator()

    if method == "temperature":
        assert raw_kind == "probability"
        logits = logit(clip_probability(raw))

        def objective(log_temperature: float) -> float:
            temperature = math.exp(log_temperature)
            probability = expit(logits / temperature)
            return float(np.mean((probability - y) ** 2))

        result = minimize_scalar(
            objective,
            bounds=(math.log(0.20), math.log(5.0)),
            method="bounded",
        )
        return TemperatureCalibrator(math.exp(float(result.x)))

    if method == "platt":
        estimator = LogisticRegression(
            C=1e3,
            solver="lbfgs",
            max_iter=5000,
        ).fit(transform_probability_logit(raw), y)
        return SklearnCalibrator(
            estimator, transform_probability_logit
        )

    if method == "beta":
        estimator = LogisticRegression(
            C=1e3,
            solver="lbfgs",
            max_iter=5000,
        ).fit(transform_beta(raw), y)
        return SklearnCalibrator(estimator, transform_beta)

    if method == "spline":
        estimator = Pipeline(
            [
                (
                    "spline",
                    SplineTransformer(
                        n_knots=4,
                        degree=2,
                        include_bias=False,
                        extrapolation="linear",
                    ),
                ),
                (
                    "logistic",
                    LogisticRegression(
                        C=10.0,
                        solver="lbfgs",
                        max_iter=5000,
                    ),
                ),
            ]
        ).fit(transform_probability_logit(raw), y)
        return SklearnCalibrator(
            estimator, transform_probability_logit
        )

    if method == "isotonic":
        if len(raw) < 100 or np.unique(raw).size < 10:
            raise ValueError(
                "Insufficient rows or unique scores for isotonic."
            )
        estimator = IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            increasing=True,
            out_of_bounds="clip",
        ).fit(raw, y)
        return IsotonicCalibrator(estimator)

    if method == "raw_platt":
        estimator = LogisticRegression(
            C=1e3,
            solver="lbfgs",
            max_iter=5000,
        ).fit(transform_raw(raw), y)
        return SklearnCalibrator(estimator, transform_raw)

    if method == "raw_spline":
        estimator = Pipeline(
            [
                (
                    "spline",
                    SplineTransformer(
                        n_knots=4,
                        degree=2,
                        include_bias=False,
                        extrapolation="linear",
                    ),
                ),
                (
                    "logistic",
                    LogisticRegression(
                        C=10.0,
                        solver="lbfgs",
                        max_iter=5000,
                    ),
                ),
            ]
        ).fit(transform_raw(raw), y)
        return SklearnCalibrator(estimator, transform_raw)

    if method == "raw_isotonic":
        if len(raw) < 100 or np.unique(raw).size < 10:
            raise ValueError(
                "Insufficient rows or unique margins for isotonic."
            )
        estimator = IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            increasing=True,
            out_of_bounds="clip",
        ).fit(raw, y)
        return IsotonicCalibrator(estimator)

    raise KeyError(f"Unknown calibration method: {method}")


CALIBRATOR_COMPLEXITY = {
    "identity": 0,
    "temperature": 1,
    "platt": 2,
    "raw_platt": 2,
    "beta": 3,
    "spline": 4,
    "raw_spline": 4,
    "isotonic": 5,
    "raw_isotonic": 5,
}


def fallback_probability(
    raw: np.ndarray,
    *,
    raw_kind: str,
) -> np.ndarray:
    raw = np.asarray(raw, dtype=float)
    if raw_kind == "probability":
        return clip_probability(raw)
    # Ten points is a conservative fixed scale for a temporary
    # cross-fitting fallback; it is never the final selected mapping.
    return clip_probability(expit(raw / 10.0))


@dataclass
class CalibrationSelection:
    method: str
    fitted_calibrator: Any
    audit: pd.DataFrame
    cross_fitted_predictions: np.ndarray


def select_cross_fitted_calibrator(
    inner_oof: pd.DataFrame,
    *,
    raw_kind: str,
) -> CalibrationSelection:
    methods = (
        MODEL_CONFIG["calibration"]["probability_methods"]
        if raw_kind == "probability"
        else MODEL_CONFIG["calibration"]["margin_methods"]
    )
    seasons = sorted(map(int, inner_oof["Season"].unique()))
    y_all = inner_oof["Team1Win"].to_numpy(dtype=int)
    raw_all = inner_oof["RawPrediction"].to_numpy(dtype=float)

    method_predictions: dict[str, np.ndarray] = {}
    records: list[dict[str, Any]] = []

    for method in methods:
        cross_fitted = np.full(len(inner_oof), np.nan, dtype=float)
        method_failed = False
        failure_message = ""
        for season in seasons:
            validation_mask = (
                inner_oof["Season"].to_numpy(dtype=int) == season
            )
            training_mask = ~validation_mask
            raw_train = raw_all[training_mask]
            y_train = y_all[training_mask]
            raw_validation = raw_all[validation_mask]
            if (
                training_mask.sum() < 40
                or np.unique(y_train).size < 2
            ):
                cross_fitted[validation_mask] = fallback_probability(
                    raw_validation,
                    raw_kind=raw_kind,
                )
                continue
            try:
                calibrator = fit_calibrator(
                    method,
                    raw_train,
                    y_train,
                    raw_kind=raw_kind,
                )
                cross_fitted[validation_mask] = calibrator.predict(
                    raw_validation
                )
            except Exception as exc:
                method_failed = True
                failure_message = repr(exc)
                cross_fitted[validation_mask] = fallback_probability(
                    raw_validation,
                    raw_kind=raw_kind,
                )

        scored = inner_oof[
            ["TargetKey", "Gender", "Season", "Team1Win"]
        ].copy()
        scored["Prediction"] = cross_fitted
        metrics, season_scores = evaluate_probability_frame(scored)
        records.append(
            {
                "Method": method,
                "RawKind": raw_kind,
                "MacroSeasonBrier": metrics["MacroSeasonBrier"],
                "GameWeightedBrier": metrics["GameWeightedBrier"],
                "SeasonBrierStd": metrics["SeasonBrierStd"],
                "WorstSeasonBrier": metrics["WorstSeasonBrier"],
                "ComplexityRank": CALIBRATOR_COMPLEXITY[method],
                "HadFallback": method_failed,
                "FailureMessage": failure_message,
            }
        )
        method_predictions[method] = cross_fitted

    audit = pd.DataFrame(records).sort_values(
        ["MacroSeasonBrier", "ComplexityRank"]
    ).reset_index(drop=True)
    best_method = str(audit.iloc[0]["Method"])
    best_predictions = method_predictions[best_method]

    best_season_scores = season_metric_values(
        pd.DataFrame(
            {
                "Season": inner_oof["Season"],
                "Team1Win": inner_oof["Team1Win"],
                "Prediction": best_predictions,
            }
        ),
        "Prediction",
    )
    standard_error = (
        float(
            best_season_scores["Brier"].std(ddof=1)
            / math.sqrt(len(best_season_scores))
        )
        if len(best_season_scores) > 1
        else 0.0
    )
    threshold = float(audit.iloc[0]["MacroSeasonBrier"]) + standard_error
    eligible = audit.loc[
        audit["MacroSeasonBrier"] <= threshold + 1e-12
    ].sort_values(["ComplexityRank", "MacroSeasonBrier"])
    selected_method = str(eligible.iloc[0]["Method"])

    audit["BestMeanMethod"] = audit["Method"].eq(best_method)
    audit["OneSEThreshold"] = threshold
    audit["WithinOneSE"] = (
        audit["MacroSeasonBrier"] <= threshold + 1e-12
    )
    audit["Selected"] = audit["Method"].eq(selected_method)

    fitted = fit_calibrator(
        selected_method,
        raw_all,
        y_all,
        raw_kind=raw_kind,
    )
    return CalibrationSelection(
        method=selected_method,
        fitted_calibrator=fitted,
        audit=audit,
        cross_fitted_predictions=method_predictions[selected_method],
    )


print("Calibration laboratory ready.")


## 7. Model engines with leakage-safe early stopping

The direct classifiers and margin regressors share the same fold machinery.

For XGBoost and LightGBM:

- inner validation seasons may control early stopping;
- the best iteration is recorded for each inner fold;
- the outer model is retrained on all outer-training seasons using the median inner best iteration;
- the outer validation season is never passed as an evaluation set.

This distinction is critical. Using the outer holdout for early stopping would make the reported outer score part of model fitting.


In [ ]:
@dataclass(frozen=True)
class ModelSpec:
    name: str
    universe: str
    task: str
    raw_kind: str
    selector_family: str
    tune: bool
    complexity_rank: int
    description: str


MODEL_SPECS: dict[str, ModelSpec] = {
    "constant_probability": ModelSpec(
        "constant_probability",
        "compact",
        "classification",
        "probability",
        "baseline",
        False,
        0,
        "Constant 0.50 absolute baseline.",
    ),
    "seed_logistic": ModelSpec(
        "seed_logistic",
        "compact",
        "classification",
        "probability",
        "baseline",
        False,
        1,
        "Seed and prequential seed-prior logistic baseline.",
    ),
    "elo_seed_logistic": ModelSpec(
        "elo_seed_logistic",
        "compact",
        "classification",
        "probability",
        "baseline",
        False,
        2,
        "Corrected current-season Elo plus seed logistic baseline.",
    ),
    "elastic_logistic": ModelSpec(
        "elastic_logistic",
        "any",
        "classification",
        "probability",
        "linear",
        True,
        3,
        "Block-selected elastic-net logistic regression.",
    ),
    "hist_classifier": ModelSpec(
        "hist_classifier",
        "rich",
        "classification",
        "probability",
        "tree",
        False,
        4,
        "CPU-safe histogram gradient-boosting challenger.",
    ),
    "xgb_classifier": ModelSpec(
        "xgb_classifier",
        "rich",
        "classification",
        "probability",
        "tree",
        True,
        5,
        "Shallow regularized XGBoost probability model.",
    ),
    "lgb_classifier": ModelSpec(
        "lgb_classifier",
        "rich",
        "classification",
        "probability",
        "tree",
        True,
        6,
        "Shallow deterministic LightGBM probability model.",
    ),
    "ridge_margin": ModelSpec(
        "ridge_margin",
        "rich",
        "margin",
        "margin",
        "margin",
        False,
        7,
        "Regularized linear point-margin model.",
    ),
    "xgb_margin": ModelSpec(
        "xgb_margin",
        "rich",
        "margin",
        "margin",
        "margin",
        True,
        8,
        "Shallow XGBoost point-margin model.",
    ),
    "lgb_margin": ModelSpec(
        "lgb_margin",
        "rich",
        "margin",
        "margin",
        "margin",
        True,
        9,
        "Shallow LightGBM point-margin model.",
    ),
    "torch_mlp": ModelSpec(
        "torch_mlp",
        "rich",
        "classification",
        "probability",
        "linear",
        True,
        10,
        "Optional small regularized CPU PyTorch neural challenger.",
    ),
}


def model_plan_for_outer(outer: pd.Series) -> list[str]:
    universe = str(outer["Universe"])
    if universe == "compact":
        return [
            "constant_probability",
            "seed_logistic",
            "elo_seed_logistic",
            "elastic_logistic",
        ]

    plan = [
        "constant_probability",
        "seed_logistic",
        "elo_seed_logistic",
        "elastic_logistic",
    ]
    if MODE["run_hist_classifier"]:
        plan.append("hist_classifier")
    plan.extend(
        [
            "xgb_classifier",
            "lgb_classifier",
            "ridge_margin",
            "xgb_margin",
        ]
    )
    if MODE["run_lgb_margin"]:
        plan.append("lgb_margin")
    if MODE["run_neural"]:
        plan.append("torch_mlp")
    return plan


def default_parameters(spec: ModelSpec) -> dict[str, Any]:
    defaults = {
        "constant_probability": {},
        "seed_logistic": {"C": 0.50},
        "elo_seed_logistic": {"C": 0.50},
        "elastic_logistic": {"C": 0.10, "l1_ratio": 0.25},
        "hist_classifier": {
            "learning_rate": 0.04,
            "max_leaf_nodes": 15,
            "max_depth": 4,
            "min_samples_leaf": 25,
            "l2_regularization": 5.0,
            "max_iter": 300,
        },
        "xgb_classifier": {
            "eta": 0.04,
            "max_depth": 3,
            "min_child_weight": 8.0,
            "subsample": 0.85,
            "colsample_bytree": 0.70,
            "reg_alpha": 0.10,
            "reg_lambda": 12.0,
            "gamma": 0.05,
            "max_bin": 128,
        },
        "lgb_classifier": {
            "learning_rate": 0.03,
            "num_leaves": 15,
            "max_depth": 4,
            "min_data_in_leaf": 35,
            "feature_fraction": 0.70,
            "bagging_fraction": 0.85,
            "bagging_freq": 1,
            "lambda_l1": 0.10,
            "lambda_l2": 12.0,
            "min_gain_to_split": 0.02,
            "max_bin": 127,
        },
        "ridge_margin": {"alpha": 100.0},
        "xgb_margin": {
            "eta": 0.035,
            "max_depth": 3,
            "min_child_weight": 8.0,
            "subsample": 0.85,
            "colsample_bytree": 0.70,
            "reg_alpha": 0.05,
            "reg_lambda": 15.0,
            "gamma": 0.0,
            "max_bin": 128,
        },
        "lgb_margin": {
            "learning_rate": 0.03,
            "num_leaves": 15,
            "max_depth": 4,
            "min_data_in_leaf": 35,
            "feature_fraction": 0.70,
            "bagging_fraction": 0.85,
            "bagging_freq": 1,
            "lambda_l1": 0.05,
            "lambda_l2": 15.0,
            "min_gain_to_split": 0.0,
            "max_bin": 127,
        },
        "torch_mlp": {
            "hidden_width": 64,
            "dropout": 0.15,
            "learning_rate": 1e-3,
            "weight_decay": 1e-3,
            "batch_size": 64,
        },
    }
    return dict(defaults[spec.name])


def sample_parameters(
    trial: optuna.Trial,
    spec: ModelSpec,
) -> dict[str, Any]:
    if spec.name == "elastic_logistic":
        return {
            "C": trial.suggest_float("C", 0.01, 2.0, log=True),
            "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 0.85),
        }
    if spec.name in {"xgb_classifier", "xgb_margin"}:
        return {
            "eta": trial.suggest_float("eta", 0.015, 0.08, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 4),
            "min_child_weight": trial.suggest_float(
                "min_child_weight", 2.0, 20.0, log=True
            ),
            "subsample": trial.suggest_float("subsample", 0.70, 1.0),
            "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.45, 0.90
            ),
            "reg_alpha": trial.suggest_float(
                "reg_alpha", 1e-4, 2.0, log=True
            ),
            "reg_lambda": trial.suggest_float(
                "reg_lambda", 3.0, 50.0, log=True
            ),
            "gamma": trial.suggest_float("gamma", 0.0, 0.50),
            "max_bin": trial.suggest_categorical(
                "max_bin", [64, 128, 256]
            ),
        }
    if spec.name in {"lgb_classifier", "lgb_margin"}:
        return {
            "learning_rate": trial.suggest_float(
                "learning_rate", 0.015, 0.07, log=True
            ),
            "num_leaves": trial.suggest_categorical(
                "num_leaves", [7, 11, 15, 23]
            ),
            "max_depth": trial.suggest_int("max_depth", 3, 6),
            "min_data_in_leaf": trial.suggest_int(
                "min_data_in_leaf", 20, 90
            ),
            "feature_fraction": trial.suggest_float(
                "feature_fraction", 0.50, 0.90
            ),
            "bagging_fraction": trial.suggest_float(
                "bagging_fraction", 0.70, 1.0
            ),
            "bagging_freq": 1,
            "lambda_l1": trial.suggest_float(
                "lambda_l1", 1e-4, 2.0, log=True
            ),
            "lambda_l2": trial.suggest_float(
                "lambda_l2", 3.0, 50.0, log=True
            ),
            "min_gain_to_split": trial.suggest_float(
                "min_gain_to_split", 0.0, 0.20
            ),
            "max_bin": trial.suggest_categorical(
                "max_bin", [63, 127, 255]
            ),
        }
    if spec.name == "torch_mlp":
        return {
            "hidden_width": trial.suggest_categorical(
                "hidden_width", [32, 64, 96]
            ),
            "dropout": trial.suggest_float("dropout", 0.05, 0.35),
            "learning_rate": trial.suggest_float(
                "learning_rate", 2e-4, 3e-3, log=True
            ),
            "weight_decay": trial.suggest_float(
                "weight_decay", 1e-5, 1e-2, log=True
            ),
            "batch_size": trial.suggest_categorical(
                "batch_size", [32, 64, 128]
            ),
        }
    return default_parameters(spec)


def prepare_numeric_matrix(
    frame: pd.DataFrame,
    features: Sequence[str],
) -> np.ndarray:
    # Always materialize a writable C-contiguous float32 array. pandas 3 uses
    # copy-on-write semantics and may otherwise return a read-only view.
    return np.array(
        frame[list(features)].to_numpy(
            dtype=np.float32,
            na_value=np.nan,
        ),
        dtype=np.float32,
        copy=True,
        order="C",
    )



DIRECTIONAL_INTERACTION_TOKENS = (
    "_edge",
    "squared_signed",
)


def mirror_feature_frame(
    frame: pd.DataFrame,
    features: Sequence[str],
    *,
    flip_targets: bool,
) -> pd.DataFrame:
    """Create the algebraic Team2-vs-Team1 representation.

    Notebook 02 already audited which feature families are directional
    versus symmetric. This transformation preserves that convention
    without materializing a second permanent feature store.
    """
    mirrored = frame.copy()
    feature_set = set(features)

    # Swap explicit team seed columns before other transformations.
    if {
        "matchup__team1_seed",
        "matchup__team2_seed",
    }.issubset(feature_set):
        mirrored[
            ["matchup__team1_seed", "matchup__team2_seed"]
        ] = frame[
            ["matchup__team2_seed", "matchup__team1_seed"]
        ].to_numpy()

    equal_seed = (
        frame["matchup__equal_seed"].fillna(0).astype(bool)
        if "matchup__equal_seed" in frame.columns
        else pd.Series(False, index=frame.index)
    )

    for feature in features:
        if feature not in frame.columns:
            continue
        if feature in {
            "matchup__team1_seed",
            "matchup__team2_seed",
        }:
            continue
        if feature.startswith("diff__"):
            mirrored[feature] = -pd.to_numeric(
                frame[feature], errors="coerce"
            )
        elif feature == "matchup__seed_diff":
            mirrored[feature] = -pd.to_numeric(
                frame[feature], errors="coerce"
            )
        elif (
            feature.startswith("seedprior__")
            and "p_team1" in feature
        ):
            mirrored[feature] = 1.0 - pd.to_numeric(
                frame[feature], errors="coerce"
            )
        elif feature == "matchup__team1_is_seed_favorite":
            original = pd.to_numeric(
                frame[feature], errors="coerce"
            )
            mirrored[feature] = np.where(
                equal_seed,
                original,
                1.0 - original,
            )
        elif feature.startswith("interaction__") and any(
            token in feature
            for token in DIRECTIONAL_INTERACTION_TOKENS
        ):
            mirrored[feature] = -pd.to_numeric(
                frame[feature], errors="coerce"
            )
        # absdiff__, mean__, availability, context, gaps, volatility,
        # and other explicitly symmetric fields remain unchanged.

    if flip_targets:
        if "Team1Win" in mirrored.columns:
            mirrored["Team1Win"] = (
                1 - mirrored["Team1Win"].astype(int)
            )
        if "Team1Margin" in mirrored.columns:
            mirrored["Team1Margin"] = -pd.to_numeric(
                mirrored["Team1Margin"], errors="coerce"
            )
        if {
            "Team1ID",
            "Team2ID",
        }.issubset(mirrored.columns):
            mirrored[["Team1ID", "Team2ID"]] = frame[
                ["Team2ID", "Team1ID"]
            ].to_numpy()
        if "TargetKey" in mirrored.columns:
            mirrored["TargetKey"] = (
                mirrored["TargetKey"].astype(str) + "__mirror"
            )
    return mirrored


def augment_training_symmetrically(
    training: pd.DataFrame,
    features: Sequence[str],
) -> pd.DataFrame:
    mirrored = mirror_feature_frame(
        training,
        features,
        flip_targets=True,
    )
    augmented = pd.concat(
        [training, mirrored],
        ignore_index=True,
    )
    return augmented


@dataclass
class FittedPredictor:
    model_name: str
    model: Any
    preprocessor: Any
    features: list[str]
    best_iteration: int | None
    raw_kind: str
    enforce_symmetry: bool = True

    def _predict_once(self, frame: pd.DataFrame) -> np.ndarray:
        if self.model_name == "constant_probability":
            return self.model.predict(np.empty(len(frame)))

        matrix = prepare_numeric_matrix(frame, self.features)
        if self.model_name in {
            "seed_logistic",
            "elo_seed_logistic",
            "elastic_logistic",
            "ridge_margin",
        }:
            transformed = self.preprocessor.transform(matrix)
            if self.raw_kind == "probability":
                return clip_probability(
                    self.model.predict_proba(transformed)[:, 1]
                )
            return np.asarray(
                self.model.predict(transformed), dtype=float
            )
        if self.model_name == "hist_classifier":
            return clip_probability(
                self.model.predict_proba(matrix)[:, 1]
            )
        if self.model_name.startswith("xgb_"):
            data = xgb.DMatrix(
                matrix,
                feature_names=self.features,
            )
            iteration = (
                self.best_iteration
                if self.best_iteration is not None
                else int(self.model.num_boosted_rounds())
            )
            return np.asarray(
                self.model.predict(
                    data,
                    iteration_range=(0, max(1, iteration)),
                ),
                dtype=float,
            )
        if self.model_name.startswith("lgb_"):
            return np.asarray(
                self.model.predict(
                    matrix,
                    num_iteration=self.best_iteration,
                ),
                dtype=float,
            )
        if self.model_name == "torch_mlp":
            transformed = self.preprocessor.transform(matrix).astype(
                np.float32
            )
            tensor = torch.from_numpy(transformed)
            self.model.eval()
            with torch.no_grad():
                logits = self.model(tensor).squeeze(1).cpu().numpy()
            return clip_probability(expit(logits))
        raise KeyError(self.model_name)

    def predict_raw(self, frame: pd.DataFrame) -> np.ndarray:
        original = self._predict_once(frame)
        if (
            not self.enforce_symmetry
            or self.model_name == "constant_probability"
        ):
            return (
                clip_probability(original)
                if self.raw_kind == "probability"
                else np.asarray(original, dtype=float)
            )
        mirrored_frame = mirror_feature_frame(
            frame,
            self.features,
            flip_targets=False,
        )
        mirrored = self._predict_once(mirrored_frame)
        if self.raw_kind == "probability":
            return clip_probability(
                0.5 * (original + (1.0 - mirrored))
            )
        return 0.5 * (
            np.asarray(original, dtype=float)
            - np.asarray(mirrored, dtype=float)
        )


def xgb_brier_metric(
    prediction: np.ndarray,
    data: xgb.DMatrix,
) -> tuple[str, float]:
    y = data.get_label()
    return "brier", float(np.mean((prediction - y) ** 2))


def lgb_brier_metric(
    prediction: np.ndarray,
    data: lgb.Dataset,
) -> tuple[str, float, bool]:
    y = data.get_label()
    return "brier", float(np.mean((prediction - y) ** 2)), False


def fit_predictor(
    spec: ModelSpec,
    params: dict[str, Any],
    training: pd.DataFrame,
    validation: pd.DataFrame | None,
    features: Sequence[str],
    *,
    fixed_iterations: int | None = None,
    random_seed: int = SEED,
) -> tuple[FittedPredictor, np.ndarray | None]:
    features = list(features)
    if spec.name != "constant_probability":
        training = augment_training_symmetrically(
            training,
            features,
        )
    y_win = training["Team1Win"].to_numpy(dtype=int)
    y_margin = training["Team1Margin"].to_numpy(dtype=np.float32)
    x_train = prepare_numeric_matrix(training, features) if features else None
    x_validation = (
        prepare_numeric_matrix(validation, features)
        if validation is not None and features
        else None
    )

    if spec.name == "constant_probability":
        probability = 0.5
        fitted = FittedPredictor(
            model_name=spec.name,
            model=ConstantCalibrator(probability),
            preprocessor=None,
            features=[],
            best_iteration=None,
            raw_kind="probability",
        )
        raw = (
            np.full(len(validation), probability)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name in {
        "seed_logistic",
        "elo_seed_logistic",
        "elastic_logistic",
    }:
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        transformed = scaler.fit_transform(
            imputer.fit_transform(x_train)
        )
        if spec.name == "elastic_logistic":
            model = LogisticRegression(
                C=float(params["C"]),
                l1_ratio=float(params["l1_ratio"]),
                solver="saga",
                max_iter=8000,
                tol=1e-4,
                random_state=random_seed,
            )
        else:
            model = LogisticRegression(
                C=float(params["C"]),
                l1_ratio=0.0,
                solver="lbfgs",
                max_iter=5000,
                random_state=random_seed,
            )
        model.fit(transformed, y_win)
        preprocessor = Pipeline(
            [("imputer", imputer), ("scaler", scaler)]
        )
        fitted = FittedPredictor(
            spec.name,
            model,
            preprocessor,
            features,
            None,
            "probability",
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name == "ridge_margin":
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        transformed = scaler.fit_transform(
            imputer.fit_transform(x_train)
        )
        model = Ridge(alpha=float(params["alpha"]))
        model.fit(transformed, y_margin)
        preprocessor = Pipeline(
            [("imputer", imputer), ("scaler", scaler)]
        )
        fitted = FittedPredictor(
            spec.name,
            model,
            preprocessor,
            features,
            None,
            "margin",
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name == "hist_classifier":
        model = HistGradientBoostingClassifier(
            **params,
            early_stopping=False,
            random_state=random_seed,
        )
        model.fit(x_train, y_win)
        fitted = FittedPredictor(
            spec.name,
            model,
            None,
            features,
            int(params["max_iter"]),
            "probability",
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name.startswith("xgb_"):
        is_classification = spec.task == "classification"
        objective = (
            "binary:logistic"
            if is_classification
            else "reg:squarederror"
        )
        xgb_params = {
            "objective": objective,
            "tree_method": "hist",
            "device": "cpu",
            "nthread": MAX_THREADS,
            "seed": random_seed,
            "verbosity": 0,
            **params,
        }
        if is_classification:
            xgb_params["disable_default_eval_metric"] = 1
        else:
            xgb_params["eval_metric"] = "rmse"

        train_matrix = xgb.QuantileDMatrix(
            x_train,
            label=y_win if is_classification else y_margin,
            feature_names=features,
            max_bin=int(params["max_bin"]),
        )
        evaluations = []
        valid_matrix = None
        callbacks = []
        train_kwargs: dict[str, Any] = {}
        num_rounds = int(fixed_iterations or 1200)

        if validation is not None and fixed_iterations is None:
            valid_label = (
                validation["Team1Win"].to_numpy(dtype=int)
                if is_classification
                else validation["Team1Margin"].to_numpy(
                    dtype=np.float32
                )
            )
            valid_matrix = xgb.QuantileDMatrix(
                x_validation,
                label=valid_label,
                feature_names=features,
                ref=train_matrix,
                max_bin=int(params["max_bin"]),
            )
            evaluations = [(valid_matrix, "valid")]
            train_kwargs["early_stopping_rounds"] = 60
        if is_classification:
            train_kwargs["custom_metric"] = xgb_brier_metric

        booster = xgb.train(
            xgb_params,
            train_matrix,
            num_boost_round=num_rounds,
            evals=evaluations,
            verbose_eval=False,
            **train_kwargs,
        )
        if (
            validation is not None
            and fixed_iterations is None
            and getattr(booster, "best_iteration", None) is not None
        ):
            best_iteration = int(booster.best_iteration) + 1
        else:
            best_iteration = int(num_rounds)
        booster.set_attr(
            max_bin_for_prediction=str(int(params["max_bin"]))
        )
        fitted = FittedPredictor(
            spec.name,
            booster,
            None,
            features,
            best_iteration,
            spec.raw_kind,
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name.startswith("lgb_"):
        is_classification = spec.task == "classification"
        lgb_params = {
            "objective": "binary" if is_classification else "regression",
            "metric": "None" if is_classification else "rmse",
            "verbosity": -1,
            "num_threads": MAX_THREADS,
            "seed": random_seed,
            "feature_fraction_seed": random_seed,
            "bagging_seed": random_seed,
            "data_random_seed": random_seed,
            "deterministic": True,
            "force_col_wise": True,
            **params,
        }
        train_set = lgb.Dataset(
            x_train,
            label=y_win if is_classification else y_margin,
            feature_name=features,
            free_raw_data=False,
        )
        callbacks = [lgb.log_evaluation(period=0)]
        valid_sets = None
        valid_names = None
        num_rounds = int(fixed_iterations or 1500)
        feval = lgb_brier_metric if is_classification else None

        if validation is not None and fixed_iterations is None:
            valid_label = (
                validation["Team1Win"].to_numpy(dtype=int)
                if is_classification
                else validation["Team1Margin"].to_numpy(
                    dtype=np.float32
                )
            )
            valid_set = lgb.Dataset(
                x_validation,
                label=valid_label,
                feature_name=features,
                reference=train_set,
                free_raw_data=False,
            )
            valid_sets = [valid_set]
            valid_names = ["valid"]
            callbacks.append(
                lgb.early_stopping(
                    stopping_rounds=60,
                    first_metric_only=True,
                    verbose=False,
                )
            )

        booster = lgb.train(
            lgb_params,
            train_set,
            num_boost_round=num_rounds,
            valid_sets=valid_sets,
            valid_names=valid_names,
            feval=feval,
            callbacks=callbacks,
        )
        best_iteration = int(
            booster.best_iteration
            if booster.best_iteration
            else num_rounds
        )
        fitted = FittedPredictor(
            spec.name,
            booster,
            None,
            features,
            best_iteration,
            spec.raw_kind,
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    if spec.name == "torch_mlp":
        if torch is None:
            raise ImportError("PyTorch is unavailable.")
        torch.set_num_threads(MAX_THREADS)
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        x_train_scaled = scaler.fit_transform(
            imputer.fit_transform(x_train)
        ).astype(np.float32)
        preprocessor = Pipeline(
            [("imputer", imputer), ("scaler", scaler)]
        )
        x_valid_scaled = (
            preprocessor.transform(x_validation).astype(np.float32)
            if validation is not None
            else None
        )

        hidden = int(params["hidden_width"])
        model = nn.Sequential(
            nn.Linear(x_train_scaled.shape[1], hidden),
            nn.ReLU(),
            nn.BatchNorm1d(hidden),
            nn.Dropout(float(params["dropout"])),
            nn.Linear(hidden, max(16, hidden // 2)),
            nn.ReLU(),
            nn.Dropout(float(params["dropout"])),
            nn.Linear(max(16, hidden // 2), 1),
        )
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=float(params["learning_rate"]),
            weight_decay=float(params["weight_decay"]),
        )
        loss_function = nn.BCEWithLogitsLoss()
        dataset = TensorDataset(
            torch.from_numpy(x_train_scaled),
            torch.from_numpy(y_win.astype(np.float32)).reshape(-1, 1),
        )
        generator = torch.Generator().manual_seed(random_seed)
        loader = DataLoader(
            dataset,
            batch_size=int(params["batch_size"]),
            shuffle=True,
            generator=generator,
            num_workers=0,
        )
        maximum_epochs = int(fixed_iterations or 400)
        best_state = None
        best_loss = float("inf")
        best_epoch = maximum_epochs
        stale = 0

        for epoch in range(1, maximum_epochs + 1):
            model.train()
            for batch_x, batch_y in loader:
                optimizer.zero_grad(set_to_none=True)
                logits = model(batch_x)
                loss = loss_function(logits, batch_y)
                loss.backward()
                optimizer.step()

            if (
                validation is not None
                and fixed_iterations is None
                and x_valid_scaled is not None
            ):
                model.eval()
                with torch.no_grad():
                    valid_logits = model(
                        torch.from_numpy(x_valid_scaled)
                    ).squeeze(1)
                    valid_probability = torch.sigmoid(
                        valid_logits
                    ).cpu().numpy()
                valid_y = validation["Team1Win"].to_numpy(dtype=float)
                valid_brier = float(
                    np.mean((valid_probability - valid_y) ** 2)
                )
                if valid_brier < best_loss - 1e-6:
                    best_loss = valid_brier
                    best_epoch = epoch
                    best_state = {
                        key: value.detach().cpu().clone()
                        for key, value in model.state_dict().items()
                    }
                    stale = 0
                else:
                    stale += 1
                if stale >= 35:
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        fitted = FittedPredictor(
            spec.name,
            model,
            preprocessor,
            features,
            int(best_epoch),
            "probability",
        )
        raw = (
            fitted.predict_raw(validation)
            if validation is not None
            else None
        )
        return fitted, raw

    raise KeyError(f"Unsupported model: {spec.name}")


print("Model engines ready:", sorted(MODEL_SPECS))


## 8. Nested hyperparameter search and inner out-of-fold generation

Optuna is used with:

- deterministic TPE sampling;
- median pruning after completed inner seasons;
- one worker, so CPU and RAM use remain predictable;
- SQLite persistence, so studies resume after interruption;
- a robust objective that penalizes unstable season-to-season performance.

The search space is intentionally shallow and strongly regularized. Tournament labels are scarce; unrestricted depth and thousands of unconstrained features would optimize noise rather than basketball signal.


In [ ]:
def rows_for_context(
    frame: pd.DataFrame,
    *,
    seasons: Sequence[int],
    gender: str,
    universe: str,
) -> pd.DataFrame:
    mask = context_mask(
        frame,
        seasons=seasons,
        gender=gender,
        universe=universe,
    )
    result = frame.loc[mask].copy()
    assert not result.empty
    assert set(map(int, result["Season"].unique())).issubset(
        set(map(int, seasons))
    )
    if gender != "Pooled":
        assert result["Gender"].eq(gender).all()
    return result


def candidates_for_context(
    *,
    gender: str,
    universe: str,
    seed_aware: bool,
) -> list[str]:
    key = candidate_set_name(
        gender=gender,
        universe=universe,
        seed_aware=seed_aware,
    )
    assert key in CANDIDATE_SETS, f"Missing candidate set: {key}"
    candidates = [
        column
        for column in CANDIDATE_SETS[key]
        if column in development.columns
        and pd.api.types.is_numeric_dtype(development[column])
    ]
    assert candidates, f"No available columns for {key}"
    return candidates


def model_feature_list(
    spec: ModelSpec,
    training: pd.DataFrame,
    candidates: Sequence[str],
    *,
    pooled: bool,
    selection_key: str,
) -> tuple[list[str], SelectionResult | None]:
    if spec.selector_family == "baseline":
        selected = baseline_features(
            spec.name,
            candidates,
            pooled=pooled,
        )
        if spec.name != "constant_probability":
            assert selected, f"No baseline features for {spec.name}"
        return selected, None

    target_column = (
        "Team1Win"
        if spec.task == "classification"
        else "Team1Margin"
    )
    result = get_selection(
        training,
        candidates,
        target_column=target_column,
        family=spec.selector_family,
        pooled=pooled,
        selection_key=selection_key,
    )
    assert len(result.selected_features) <= result.effective_cap
    return result.selected_features, result


def make_prediction_frame(
    validation: pd.DataFrame,
    *,
    outer_fold_id: str,
    inner_fold_id: str | None,
    architecture: str,
    universe: str,
    model_name: str,
    raw_prediction: np.ndarray,
    feature_count: int,
    best_iteration: int | None,
) -> pd.DataFrame:
    columns = [
        "TargetKey",
        "Gender",
        "Season",
        "Team1ID",
        "Team2ID",
        "Team1Win",
        "Team1Margin",
    ]
    output = validation[columns].copy()
    output["OuterFoldID"] = outer_fold_id
    output["InnerFoldID"] = inner_fold_id
    output["Architecture"] = architecture
    output["Universe"] = universe
    output["Model"] = model_name
    output["RawPrediction"] = np.asarray(
        raw_prediction, dtype=np.float64
    )
    output["FeatureCount"] = int(feature_count)
    output["BestIteration"] = (
        np.nan if best_iteration is None else int(best_iteration)
    )
    return output


def inner_rows_for_outer(outer_fold_id: str) -> pd.DataFrame:
    rows = inner_folds.loc[
        inner_folds["OuterFoldID"].eq(outer_fold_id)
    ].sort_values("InnerValidationSeason")
    cap = int(MODE["inner_fold_cap"])
    if len(rows) > cap:
        rows = rows.tail(cap)
    assert not rows.empty, (
        f"No inner folds available for {outer_fold_id}"
    )
    return rows.reset_index(drop=True)


def run_inner_oof(
    *,
    outer: pd.Series,
    spec: ModelSpec,
    params: dict[str, Any],
    candidates: Sequence[str],
    collect_selector_audits: bool,
    trial: optuna.Trial | None = None,
) -> tuple[pd.DataFrame, list[int], list[pd.DataFrame]]:
    predictions: list[pd.DataFrame] = []
    best_iterations: list[int] = []
    selector_audits: list[pd.DataFrame] = []
    inner_rows = inner_rows_for_outer(str(outer["OuterFoldID"]))
    gender = str(outer["Gender"])
    universe = str(outer["Universe"])
    pooled = gender == "Pooled"

    for step, (_, inner) in enumerate(
        inner_rows.iterrows(), start=1
    ):
        training_seasons = parse_json_int_list(
            inner["InnerTrainingSeasonsJSON"]
        )
        validation_season = int(inner["InnerValidationSeason"])
        assert max(training_seasons) < validation_season
        assert validation_season < int(outer["ValidationSeason"])

        training = rows_for_context(
            development,
            seasons=training_seasons,
            gender=gender,
            universe=universe,
        )
        validation = rows_for_context(
            development,
            seasons=[validation_season],
            gender=gender,
            universe=universe,
        )
        assert training["Season"].max() < validation_season
        assert validation["Season"].eq(validation_season).all()

        selection_key = (
            f"{inner['InnerFoldID']}__{spec.selector_family}"
            f"__{spec.task}"
        )
        features, selection = model_feature_list(
            spec,
            training,
            candidates,
            pooled=pooled,
            selection_key=selection_key,
        )
        if selection is not None and collect_selector_audits:
            audit = selection.audit.loc[
                selection.audit["Selected"]
            ].copy()
            audit["OuterFoldID"] = outer["OuterFoldID"]
            audit["InnerFoldID"] = inner["InnerFoldID"]
            audit["Model"] = spec.name
            audit["TargetColumn"] = (
                "Team1Win"
                if spec.task == "classification"
                else "Team1Margin"
            )
            selector_audits.append(audit)

        with threadpool_limits(limits=MAX_THREADS):
            fitted, raw = fit_predictor(
                spec,
                params,
                training,
                validation,
                features,
                fixed_iterations=None,
                random_seed=SEED + validation_season,
            )
        assert raw is not None
        predictions.append(
            make_prediction_frame(
                validation,
                outer_fold_id=str(outer["OuterFoldID"]),
                inner_fold_id=str(inner["InnerFoldID"]),
                architecture=str(outer["Architecture"]),
                universe=universe,
                model_name=spec.name,
                raw_prediction=raw,
                feature_count=len(features),
                best_iteration=fitted.best_iteration,
            )
        )
        if fitted.best_iteration is not None:
            best_iterations.append(int(fitted.best_iteration))

        if trial is not None:
            partial = pd.concat(predictions, ignore_index=True)
            score = (
                robust_probability_objective(
                    partial.assign(
                        Prediction=clip_probability(
                            partial["RawPrediction"]
                        )
                    )
                )
                if spec.task == "classification"
                else robust_margin_objective(partial)
            )
            trial.report(float(score), step=step)
            if trial.should_prune():
                raise optuna.TrialPruned()

        del training, validation, fitted, raw
        gc.collect()

    inner_oof = pd.concat(predictions, ignore_index=True)
    assert inner_oof["TargetKey"].is_unique
    return inner_oof, best_iterations, selector_audits


def tune_model(
    *,
    outer: pd.Series,
    spec: ModelSpec,
    candidates: Sequence[str],
) -> tuple[dict[str, Any], list[int], dict[str, Any]]:
    if not spec.tune:
        return default_parameters(spec), [], {
            "SearchStrategy": "fixed_preregistered",
            "TrialsRequested": 0,
            "TrialsCompleted": 0,
        }

    requested = int(MODE["trial_budgets"].get(spec.name, 0))
    if requested <= 0:
        return default_parameters(spec), [], {
            "SearchStrategy": "fixed_due_to_zero_budget",
            "TrialsRequested": 0,
            "TrialsCompleted": 0,
        }

    study_name = sanitize_name(
        f"{MODEL_CONFIG['model_contract_sha256'][:10]}"
        f"__{outer['OuterFoldID']}__{spec.name}"
        f"__{RUN_MODE}"
    )
    database_path = STUDY_DIR / f"{study_name}.sqlite3"
    storage = f"sqlite:///{database_path.as_posix()}"
    sampler = optuna.samplers.TPESampler(
        seed=SEED,
        n_startup_trials=min(3, requested),
        multivariate=True,
    )
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=min(2, requested),
        n_warmup_steps=1,
        interval_steps=1,
    )
    study = optuna.create_study(
        direction="minimize",
        study_name=study_name,
        storage=storage,
        load_if_exists=True,
        sampler=sampler,
        pruner=pruner,
    )

    def objective(trial: optuna.Trial) -> float:
        params = sample_parameters(trial, spec)
        try:
            inner_oof, iterations, _ = run_inner_oof(
                outer=outer,
                spec=spec,
                params=params,
                candidates=candidates,
                collect_selector_audits=False,
                trial=trial,
            )
            score = (
                robust_probability_objective(
                    inner_oof.assign(
                        Prediction=clip_probability(
                            inner_oof["RawPrediction"]
                        )
                    )
                )
                if spec.task == "classification"
                else robust_margin_objective(inner_oof)
            )
            trial.set_user_attr(
                "best_iterations",
                list(map(int, iterations)),
            )
            trial.set_user_attr(
                "inner_validation_seasons",
                sorted(
                    map(int, inner_oof["Season"].unique())
                ),
            )
            return float(score)
        except optuna.TrialPruned:
            raise
        except Exception as exc:
            trial.set_user_attr("failure", repr(exc))
            return float(1e6)
        finally:
            gc.collect()

    completed_or_running = len(study.trials)
    additional = max(0, requested - completed_or_running)
    if additional:
        study.optimize(
            objective,
            n_trials=additional,
            n_jobs=1,
            gc_after_trial=True,
            show_progress_bar=False,
        )

    valid_trials = [
        trial
        for trial in study.trials
        if trial.state == optuna.trial.TrialState.COMPLETE
        and np.isfinite(trial.value)
        and trial.value < 1e5
    ]
    if not valid_trials:
        warnings.warn(
            f"No valid Optuna trial for {spec.name} "
            f"{outer['OuterFoldID']}; using defaults."
        )
        return default_parameters(spec), [], {
            "SearchStrategy": "default_after_failed_search",
            "TrialsRequested": requested,
            "TrialsCompleted": 0,
        }

    best = min(valid_trials, key=lambda trial: float(trial.value))
    best_iterations = list(
        map(int, best.user_attrs.get("best_iterations", []))
    )
    metadata = {
        "SearchStrategy": "optuna_tpe_nested_inner_folds",
        "StudyName": study_name,
        "DatabasePath": str(database_path),
        "TrialsRequested": requested,
        "TrialsCompleted": len(valid_trials),
        "BestTrialNumber": int(best.number),
        "BestObjective": float(best.value),
    }
    return dict(best.params), best_iterations, metadata


print("Nested tuning utilities ready.")


## 9. Outer-fold runner with resume support

Each outer-fold result represents a complete historical deployment simulation:

```text
inner folds
→ hyperparameter search
→ inner OOF raw predictions
→ cross-fitted calibrator selection
→ outer-training feature selection
→ full outer-training refit
→ untouched outer-season prediction
```

Failure of an optional model is logged without discarding completed work. Core baseline and boosting failures remain visible in the readiness report.


In [ ]:
MODEL_FAILURES: list[dict[str, Any]] = []
MODEL_RUN_METADATA: list[dict[str, Any]] = []
SELECTED_FEATURE_RECORDS: list[pd.DataFrame] = []
CALIBRATION_RECORDS: list[pd.DataFrame] = []


def outer_model_directory(
    outer_fold_id: str,
    model_name: str,
) -> Path:
    return (
        CACHE_DIR
        / "outer_models"
        / sanitize_name(outer_fold_id)
        / sanitize_name(model_name)
    )


def load_outer_checkpoint(
    directory: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    metadata = read_json(directory / "metadata.json")
    outer_predictions = pd.read_parquet(
        directory / "outer_predictions.parquet"
    )
    inner_oof = pd.read_parquet(directory / "inner_oof.parquet")
    return outer_predictions, inner_oof, metadata


def run_outer_model(
    outer: pd.Series,
    model_name: str,
    *,
    seed_aware: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    spec = MODEL_SPECS[model_name]
    outer_fold_id = str(outer["OuterFoldID"])
    directory = outer_model_directory(
        outer_fold_id,
        model_name
        + ("" if seed_aware else "__seed_free"),
    )
    if checkpoint_is_valid(directory):
        outer_predictions, inner_oof, metadata = load_outer_checkpoint(
            directory
        )
        metadata["LoadedFromCheckpoint"] = True
        return outer_predictions, inner_oof, metadata

    gender = str(outer["Gender"])
    universe = str(outer["Universe"])
    pooled = gender == "Pooled"
    training_seasons = parse_json_int_list(
        outer["TrainingSeasonsJSON"]
    )
    validation_season = int(outer["ValidationSeason"])
    assert max(training_seasons) < validation_season
    assert validation_season <= DEVELOPMENT_LAST_SEASON

    outer_training = rows_for_context(
        development,
        seasons=training_seasons,
        gender=gender,
        universe=universe,
    )
    outer_validation = rows_for_context(
        development,
        seasons=[validation_season],
        gender=gender,
        universe=universe,
    )
    assert outer_training["Season"].max() < validation_season
    assert outer_validation["Season"].eq(validation_season).all()

    candidates = candidates_for_context(
        gender=gender,
        universe=universe,
        seed_aware=seed_aware,
    )
    if spec.universe == "rich":
        assert universe == "rich"

    start_rss = rss_mb()
    start_time = time.perf_counter()

    params, search_iterations, search_metadata = tune_model(
        outer=outer,
        spec=spec,
        candidates=candidates,
    )

    # Re-run the chosen parameterization to obtain one clean inner OOF
    # vector and selection audit. This is the input to calibration.
    inner_oof, inner_iterations, selector_audits = run_inner_oof(
        outer=outer,
        spec=spec,
        params=params,
        candidates=candidates,
        collect_selector_audits=True,
        trial=None,
    )

    if spec.name == "constant_probability":
        calibration = CalibrationSelection(
            method="identity",
            fitted_calibrator=IdentityCalibrator(),
            audit=pd.DataFrame(
                [
                    {
                        "Method": "identity",
                        "RawKind": "probability",
                        "MacroSeasonBrier": float(
                            np.mean(
                                (
                                    inner_oof["RawPrediction"].to_numpy(
                                        dtype=float
                                    )
                                    - inner_oof["Team1Win"].to_numpy(
                                        dtype=float
                                    )
                                )
                                ** 2
                            )
                        ),
                        "ComplexityRank": 0,
                        "Selected": True,
                    }
                ]
            ),
            cross_fitted_predictions=clip_probability(
                inner_oof["RawPrediction"]
            ),
        )
    else:
        calibration = select_cross_fitted_calibrator(
            inner_oof,
            raw_kind=spec.raw_kind,
        )

    inner_oof["Prediction"] = (
        calibration.cross_fitted_predictions
    )
    inner_oof["CalibrationMethod"] = calibration.method

    selection_key = (
        f"{outer_fold_id}__outer__{spec.selector_family}"
        f"__{spec.task}"
    )
    outer_features, outer_selection = model_feature_list(
        spec,
        outer_training,
        candidates,
        pooled=pooled,
        selection_key=selection_key,
    )
    if outer_selection is not None:
        selected_audit = outer_selection.audit.loc[
            outer_selection.audit["Selected"]
        ].copy()
        selected_audit["OuterFoldID"] = outer_fold_id
        selected_audit["InnerFoldID"] = None
        selected_audit["Model"] = spec.name
        selected_audit["TargetColumn"] = (
            "Team1Win"
            if spec.task == "classification"
            else "Team1Margin"
        )
        selector_audits.append(selected_audit)

    usable_iterations = inner_iterations or search_iterations
    fixed_iterations = (
        max(10, int(round(float(np.median(usable_iterations)))))
        if usable_iterations
        else None
    )

    with threadpool_limits(limits=MAX_THREADS):
        fitted, raw_outer = fit_predictor(
            spec,
            params,
            outer_training,
            outer_validation,
            outer_features,
            fixed_iterations=fixed_iterations,
            random_seed=SEED + validation_season,
        )
    assert raw_outer is not None
    calibrated_outer = calibration.fitted_calibrator.predict(
        raw_outer
    )
    outer_predictions = make_prediction_frame(
        outer_validation,
        outer_fold_id=outer_fold_id,
        inner_fold_id=None,
        architecture=str(outer["Architecture"]),
        universe=universe,
        model_name=(
            spec.name
            if seed_aware
            else f"{spec.name}__seed_free"
        ),
        raw_prediction=raw_outer,
        feature_count=len(outer_features),
        best_iteration=fitted.best_iteration,
    )
    outer_predictions["Prediction"] = calibrated_outer
    outer_predictions["CalibrationMethod"] = calibration.method

    outer_metrics, _ = evaluate_probability_frame(
        outer_predictions
    )
    elapsed = time.perf_counter() - start_time

    calibration_audit = calibration.audit.copy()
    calibration_audit["OuterFoldID"] = outer_fold_id
    calibration_audit["Model"] = spec.name
    calibration_audit["ValidationSeason"] = validation_season
    calibration_audit["SeedAware"] = bool(seed_aware)

    selected_feature_frame = (
        pd.concat(selector_audits, ignore_index=True)
        if selector_audits
        else pd.DataFrame(
            {
                "Feature": outer_features,
                "Block": [
                    FEATURE_TO_BLOCK.get(feature, "other")
                    for feature in outer_features
                ],
                "MissingRate": np.nan,
                "NonMissingRows": len(outer_training),
                "UniqueValues": [
                    int(outer_training[feature].nunique(dropna=True))
                    for feature in outer_features
                ],
                "GlobalCorrelation": np.nan,
                "MedianAbsoluteSeasonCorrelation": np.nan,
                "SeasonSignConsistency": np.nan,
                "SeasonCoverage": np.nan,
                "SeasonCorrelationIQR": np.nan,
                "StabilityScore": np.nan,
                "Mandatory": True,
                "InPrecorrelationPool": True,
                "Selected": True,
                "SelectionRank": np.arange(1, len(outer_features) + 1),
                "ExclusionReason": "preregistered_baseline",
                "OuterFoldID": outer_fold_id,
                "InnerFoldID": None,
                "Model": spec.name,
            }
        )
    )
    selected_feature_frame["SeedAware"] = bool(seed_aware)

    coefficient_frame = pd.DataFrame()
    if hasattr(fitted.model, "coef_"):
        coefficients = np.asarray(fitted.model.coef_, dtype=float).reshape(-1)
        if len(coefficients) == len(outer_features):
            coefficient_frame = pd.DataFrame(
                {
                    "OuterFoldID": outer_fold_id,
                    "ValidationSeason": validation_season,
                    "Gender": gender,
                    "Architecture": str(outer["Architecture"]),
                    "Universe": universe,
                    "Model": spec.name,
                    "Feature": outer_features,
                    "Block": [
                        FEATURE_TO_BLOCK.get(feature, "other")
                        for feature in outer_features
                    ],
                    "StandardizedCoefficient": coefficients,
                }
            )

    metadata = {
        "status": "complete",
        "contracts": EXPECTED_CONTRACTS,
        "run_mode": RUN_MODE,
        "outer_fold_id": outer_fold_id,
        "architecture": str(outer["Architecture"]),
        "universe": universe,
        "gender": gender,
        "validation_season": validation_season,
        "model": spec.name,
        "seed_aware": bool(seed_aware),
        "task": spec.task,
        "raw_kind": spec.raw_kind,
        "parameters": params,
        "fixed_iterations_from_inner": fixed_iterations,
        "inner_best_iterations": list(
            map(int, usable_iterations)
        ),
        "outer_selected_features": outer_features,
        "outer_feature_count": len(outer_features),
        "calibration_method": calibration.method,
        "outer_metrics": outer_metrics,
        "search": search_metadata,
        "elapsed_seconds": elapsed,
        "rss_start_mb": start_rss,
        "rss_end_mb": rss_mb(),
        "loaded_from_checkpoint": False,
    }

    directory.mkdir(parents=True, exist_ok=True)
    atomic_write_parquet(
        directory / "inner_oof.parquet",
        inner_oof,
    )
    atomic_write_parquet(
        directory / "outer_predictions.parquet",
        outer_predictions,
    )
    atomic_write_csv(
        directory / "selected_features.csv",
        selected_feature_frame,
    )
    atomic_write_csv(
        directory / "calibration_audit.csv",
        calibration_audit,
    )
    if not coefficient_frame.empty:
        atomic_write_csv(
            directory / "linear_coefficients.csv",
            coefficient_frame,
        )
    atomic_write_json(directory / "metadata.json", metadata)

    SELECTED_FEATURE_RECORDS.append(selected_feature_frame)
    CALIBRATION_RECORDS.append(calibration_audit)
    MODEL_RUN_METADATA.append(metadata)

    del (
        outer_training,
        outer_validation,
        fitted,
        raw_outer,
        calibration,
        inner_oof,
    )
    gc.collect()
    return (
        pd.read_parquet(directory / "outer_predictions.parquet"),
        pd.read_parquet(directory / "inner_oof.parquet"),
        metadata,
    )


def failure_record(
    *,
    outer: pd.Series,
    model_name: str,
    exc: Exception,
) -> None:
    record = {
        "OuterFoldID": str(outer["OuterFoldID"]),
        "ValidationSeason": int(outer["ValidationSeason"]),
        "Architecture": str(outer["Architecture"]),
        "Universe": str(outer["Universe"]),
        "Gender": str(outer["Gender"]),
        "Model": model_name,
        "ErrorType": type(exc).__name__,
        "Error": str(exc),
        "Traceback": traceback.format_exc(),
    }
    MODEL_FAILURES.append(record)
    atomic_write_json(
        FAILURE_DIR
        / f"{sanitize_name(record['OuterFoldID'])}"
        f"__{sanitize_name(model_name)}.json",
        record,
    )


print("Outer-fold runner ready.")


## 10. Execute the frozen outer-fold plan

The run order is deterministic:

1. compact separate-gender folds and transparent baselines;
2. rich separate-gender folds;
3. rich pooled challenger folds;
4. ensembles and partial pooling.

`standard` mode uses every frozen outer fold. `smoke` mode keeps only the most recent fold per architecture/universe/gender context, allowing a fast end-to-end verification before the full run.


In [ ]:
def build_outer_execution_plan() -> pd.DataFrame:
    plan = outer_folds.copy()
    plan = plan.loc[
        plan["ValidationSeason"].le(DEVELOPMENT_LAST_SEASON)
    ].copy()
    if MODE["outer_folds_per_context"] is not None:
        count = int(MODE["outer_folds_per_context"])
        plan = (
            plan.sort_values("ValidationSeason")
            .groupby(
                ["Universe", "Architecture", "Gender"],
                observed=True,
                group_keys=False,
            )
            .tail(count)
        )
    plan["ModelsJSON"] = plan.apply(
        lambda row: json.dumps(model_plan_for_outer(row)),
        axis=1,
    )
    return plan.sort_values(
        ["Universe", "Architecture", "Gender", "ValidationSeason"]
    ).reset_index(drop=True)


outer_plan = build_outer_execution_plan()
outer_plan.to_csv(
    MODEL_REPORTS / f"outer_execution_plan_{RUN_MODE}.csv",
    index=False,
)
print(
    outer_plan.groupby(
        ["Universe", "Architecture", "Gender"],
        observed=True,
    )
    .agg(
        Folds=("OuterFoldID", "nunique"),
        FirstSeason=("ValidationSeason", "min"),
        LastSeason=("ValidationSeason", "max"),
    )
    .reset_index()
)
print("Planned fold/model tasks:", int(
    sum(len(json.loads(value)) for value in outer_plan["ModelsJSON"])
))


In [ ]:
OUTER_PREDICTIONS: list[pd.DataFrame] = []
INNER_OOF_BY_OUTER_MODEL: dict[
    tuple[str, str], pd.DataFrame
] = {}
RUN_PROGRESS: list[dict[str, Any]] = []

total_tasks = int(
    sum(len(json.loads(value)) for value in outer_plan["ModelsJSON"])
)
completed_task_number = 0

for _, outer in outer_plan.iterrows():
    model_names = json.loads(outer["ModelsJSON"])
    for model_name in model_names:
        completed_task_number += 1
        label = (
            f"[{completed_task_number}/{total_tasks}] "
            f"{outer['OuterFoldID']} :: {model_name}"
        )
        print(label)
        try:
            ensure_memory_headroom(label)
            with ResourceTimer(
                "outer_model",
                outer_fold_id=str(outer["OuterFoldID"]),
                model_name=model_name,
            ):
                outer_prediction, inner_oof, metadata = run_outer_model(
                    outer,
                    model_name,
                    seed_aware=True,
                )
            OUTER_PREDICTIONS.append(outer_prediction)
            INNER_OOF_BY_OUTER_MODEL[
                (str(outer["OuterFoldID"]), model_name)
            ] = inner_oof
            RUN_PROGRESS.append(
                {
                    "OuterFoldID": outer["OuterFoldID"],
                    "Model": model_name,
                    "Status": "complete",
                    "LoadedFromCheckpoint": bool(
                        metadata.get(
                            "LoadedFromCheckpoint",
                            metadata.get("loaded_from_checkpoint", False),
                        )
                    ),
                    "MacroSeasonBrier": metadata[
                        "outer_metrics"
                    ]["MacroSeasonBrier"],
                    "FeatureCount": metadata[
                        "outer_feature_count"
                    ],
                    "CalibrationMethod": metadata[
                        "calibration_method"
                    ],
                }
            )
        except Exception as exc:
            failure_record(
                outer=outer,
                model_name=model_name,
                exc=exc,
            )
            RUN_PROGRESS.append(
                {
                    "OuterFoldID": outer["OuterFoldID"],
                    "Model": model_name,
                    "Status": "failed",
                    "LoadedFromCheckpoint": False,
                    "MacroSeasonBrier": np.nan,
                    "FeatureCount": np.nan,
                    "CalibrationMethod": None,
                    "Error": repr(exc),
                }
            )
            print("FAILED:", repr(exc))
        finally:
            gc.collect()

run_progress = pd.DataFrame(RUN_PROGRESS)
run_progress.to_csv(
    MODEL_REPORTS / f"run_progress_{RUN_MODE}.csv",
    index=False,
)
failure_report_path = MODEL_REPORTS / f"model_failures_{RUN_MODE}.csv"
if MODEL_FAILURES:
    pd.DataFrame(MODEL_FAILURES).to_csv(
        failure_report_path,
        index=False,
    )
elif failure_report_path.exists():
    # Remove a stale report from an earlier failed diagnostic run.
    failure_report_path.unlink()

print(
    run_progress.groupby(["Status", "Model"], observed=True)
    .size()
    .to_frame("Tasks")
    .reset_index()
)
print("Current RSS MB:", round(rss_mb(), 2))


## 11. Diversity-aware constrained ensembles

For each outer fold, the notebook uses only that fold’s inner out-of-fold predictions to:

1. rank base models by macro season Brier;
2. remove nearly duplicate prediction streams;
3. retain at most five diverse members;
4. optimize nonnegative weights that sum to one;
5. penalize concentrated weights slightly;
6. cross-fit a final ensemble calibrator by inner season;
7. apply the frozen weights and calibrator to the untouched outer season.

No fixed leaderboard-derived weights are copied from another solution.


In [ ]:
def prediction_correlation_prune(
    wide: pd.DataFrame,
    model_columns: Sequence[str],
    *,
    maximum_members: int,
    threshold: float,
) -> tuple[list[str], pd.DataFrame]:
    ranking_records = []
    for model in model_columns:
        frame = wide[
            ["TargetKey", "Season", "Team1Win", model]
        ].rename(columns={model: "Prediction"})
        metrics, _ = evaluate_probability_frame(frame)
        ranking_records.append(
            {
                "Model": model,
                "MacroSeasonBrier": metrics["MacroSeasonBrier"],
            }
        )
    ranking = pd.DataFrame(ranking_records).sort_values(
        "MacroSeasonBrier"
    )
    correlation = wide[list(model_columns)].corr().abs()

    kept: list[str] = []
    for model in ranking["Model"]:
        if not kept:
            kept.append(model)
            continue
        if all(
            float(correlation.loc[model, existing]) < threshold
            for existing in kept
        ):
            kept.append(model)
        if len(kept) >= maximum_members:
            break

    if len(kept) == 1 and len(ranking) > 1:
        # Preserve at least two members when available, even if highly
        # correlated; regularization can assign zero weight.
        kept.append(
            next(
                model
                for model in ranking["Model"]
                if model not in kept
            )
        )
    ranking["KeptForWeightOptimization"] = ranking["Model"].isin(kept)
    return kept, ranking


def macro_season_brier_from_arrays(
    y: np.ndarray,
    p: np.ndarray,
    seasons: np.ndarray,
) -> float:
    values = []
    for season in np.unique(seasons):
        mask = seasons == season
        values.append(float(np.mean((p[mask] - y[mask]) ** 2)))
    return float(np.mean(values))


def optimize_simplex_weights(
    prediction_matrix: np.ndarray,
    y: np.ndarray,
    seasons: np.ndarray,
    *,
    l2_penalty: float = 0.002,
) -> np.ndarray:
    prediction_matrix = np.asarray(
        prediction_matrix, dtype=np.float64
    )
    y = np.asarray(y, dtype=np.float64)
    seasons = np.asarray(seasons, dtype=int)
    n_models = prediction_matrix.shape[1]
    if n_models == 1:
        return np.ones(1, dtype=np.float64)

    initial = np.full(n_models, 1.0 / n_models)

    def objective(weights: np.ndarray) -> float:
        probability = clip_probability(
            prediction_matrix @ weights
        )
        return float(
            macro_season_brier_from_arrays(
                y, probability, seasons
            )
            + l2_penalty * np.sum(weights**2)
        )

    result = minimize(
        objective,
        initial,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n_models,
        constraints=[
            {
                "type": "eq",
                "fun": lambda weights: float(np.sum(weights) - 1.0),
            }
        ],
        options={"maxiter": 2000, "ftol": 1e-12},
    )
    if not result.success:
        warnings.warn(
            f"Ensemble optimizer did not converge: {result.message}. "
            "Using equal weights."
        )
        return initial
    weights = np.clip(result.x, 0.0, 1.0)
    return weights / weights.sum()


ENSEMBLE_WEIGHT_RECORDS: list[pd.DataFrame] = []
ENSEMBLE_PREDICTIONS: list[pd.DataFrame] = []
ENSEMBLE_INNER_OOF: dict[str, pd.DataFrame] = {}


def build_outer_ensemble(
    outer: pd.Series,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    outer_fold_id = str(outer["OuterFoldID"])
    directory = outer_model_directory(
        outer_fold_id, "constrained_ensemble"
    )
    if checkpoint_is_valid(directory):
        outer_prediction, inner_oof, metadata = load_outer_checkpoint(
            directory
        )
        weights = pd.read_csv(directory / "ensemble_weights.csv")
        return outer_prediction, inner_oof, weights

    available_models = [
        model_name
        for model_name in json.loads(outer["ModelsJSON"])
        if (
            outer_fold_id,
            model_name,
        ) in INNER_OOF_BY_OUTER_MODEL
        and model_name != "constant_probability"
    ]
    if len(available_models) < 2:
        raise RuntimeError(
            f"Fewer than two models available for ensemble {outer_fold_id}"
        )

    inner_base = None
    for model_name in available_models:
        frame = INNER_OOF_BY_OUTER_MODEL[
            (outer_fold_id, model_name)
        ][
            [
                "TargetKey",
                "Gender",
                "Season",
                "Team1Win",
                "Team1Margin",
                "Prediction",
            ]
        ].rename(columns={"Prediction": model_name})
        inner_base = (
            frame
            if inner_base is None
            else inner_base.merge(
                frame[
                    ["TargetKey", model_name]
                ],
                on="TargetKey",
                how="inner",
                validate="one_to_one",
            )
        )
    assert inner_base is not None

    outer_model_frames = [
        frame
        for frame in OUTER_PREDICTIONS
        if (
            frame["OuterFoldID"].iloc[0] == outer_fold_id
            and frame["Model"].iloc[0] in available_models
        )
    ]
    outer_base = None
    for frame in outer_model_frames:
        model_name = str(frame["Model"].iloc[0])
        part = frame[
            [
                "TargetKey",
                "Gender",
                "Season",
                "Team1ID",
                "Team2ID",
                "Team1Win",
                "Team1Margin",
                "Prediction",
            ]
        ].rename(columns={"Prediction": model_name})
        outer_base = (
            part
            if outer_base is None
            else outer_base.merge(
                part[["TargetKey", model_name]],
                on="TargetKey",
                how="inner",
                validate="one_to_one",
            )
        )
    assert outer_base is not None

    common_models = [
        model
        for model in available_models
        if model in inner_base.columns and model in outer_base.columns
    ]
    kept, ranking = prediction_correlation_prune(
        inner_base,
        common_models,
        maximum_members=int(
            MODEL_CONFIG["ensemble"]["maximum_members"]
        ),
        threshold=float(
            MODEL_CONFIG["ensemble"][
                "redundancy_correlation_threshold"
            ]
        ),
    )
    matrix_inner = inner_base[kept].to_numpy(dtype=np.float64)
    y_inner = inner_base["Team1Win"].to_numpy(dtype=float)
    season_inner = inner_base["Season"].to_numpy(dtype=int)
    weights = optimize_simplex_weights(
        matrix_inner,
        y_inner,
        season_inner,
    )
    raw_inner = clip_probability(matrix_inner @ weights)

    ensemble_inner = inner_base[
        ["TargetKey", "Gender", "Season", "Team1Win", "Team1Margin"]
    ].copy()
    ensemble_inner["OuterFoldID"] = outer_fold_id
    ensemble_inner["InnerFoldID"] = "ensemble_from_inner_oof"
    ensemble_inner["Architecture"] = str(outer["Architecture"])
    ensemble_inner["Universe"] = str(outer["Universe"])
    ensemble_inner["Model"] = "constrained_ensemble"
    ensemble_inner["RawPrediction"] = raw_inner
    ensemble_inner["FeatureCount"] = np.nan
    ensemble_inner["BestIteration"] = np.nan

    calibration = select_cross_fitted_calibrator(
        ensemble_inner,
        raw_kind="probability",
    )
    ensemble_inner["Prediction"] = (
        calibration.cross_fitted_predictions
    )
    ensemble_inner["CalibrationMethod"] = calibration.method

    raw_outer = clip_probability(
        outer_base[kept].to_numpy(dtype=np.float64) @ weights
    )
    outer_probability = calibration.fitted_calibrator.predict(
        raw_outer
    )
    ensemble_outer = outer_base[
        [
            "TargetKey",
            "Gender",
            "Season",
            "Team1ID",
            "Team2ID",
            "Team1Win",
            "Team1Margin",
        ]
    ].copy()
    ensemble_outer["OuterFoldID"] = outer_fold_id
    ensemble_outer["InnerFoldID"] = None
    ensemble_outer["Architecture"] = str(outer["Architecture"])
    ensemble_outer["Universe"] = str(outer["Universe"])
    ensemble_outer["Model"] = "constrained_ensemble"
    ensemble_outer["RawPrediction"] = raw_outer
    ensemble_outer["Prediction"] = outer_probability
    ensemble_outer["CalibrationMethod"] = calibration.method
    ensemble_outer["FeatureCount"] = np.nan
    ensemble_outer["BestIteration"] = np.nan

    weight_frame = pd.DataFrame(
        {
            "OuterFoldID": outer_fold_id,
            "Model": kept,
            "Weight": weights,
            "ValidationSeason": int(outer["ValidationSeason"]),
            "Architecture": str(outer["Architecture"]),
            "Universe": str(outer["Universe"]),
            "Gender": str(outer["Gender"]),
        }
    ).merge(ranking, on="Model", how="left")
    outer_metrics, _ = evaluate_probability_frame(ensemble_outer)

    metadata = {
        "status": "complete",
        "contracts": EXPECTED_CONTRACTS,
        "run_mode": RUN_MODE,
        "outer_fold_id": outer_fold_id,
        "model": "constrained_ensemble",
        "architecture": str(outer["Architecture"]),
        "universe": str(outer["Universe"]),
        "gender": str(outer["Gender"]),
        "validation_season": int(outer["ValidationSeason"]),
        "members": kept,
        "weights": dict(zip(kept, map(float, weights), strict=True)),
        "calibration_method": calibration.method,
        "outer_metrics": outer_metrics,
        "outer_feature_count": None,
        "loaded_from_checkpoint": False,
    }
    directory.mkdir(parents=True, exist_ok=True)
    atomic_write_parquet(
        directory / "inner_oof.parquet", ensemble_inner
    )
    atomic_write_parquet(
        directory / "outer_predictions.parquet", ensemble_outer
    )
    atomic_write_csv(
        directory / "ensemble_weights.csv", weight_frame
    )
    atomic_write_csv(
        directory / "calibration_audit.csv",
        calibration.audit.assign(
            OuterFoldID=outer_fold_id,
            Model="constrained_ensemble",
        ),
    )
    atomic_write_json(directory / "metadata.json", metadata)
    return ensemble_outer, ensemble_inner, weight_frame


for _, outer in outer_plan.iterrows():
    try:
        ensemble_outer, ensemble_inner, weight_frame = (
            build_outer_ensemble(outer)
        )
        ENSEMBLE_PREDICTIONS.append(ensemble_outer)
        ENSEMBLE_INNER_OOF[
            str(outer["OuterFoldID"])
        ] = ensemble_inner
        ENSEMBLE_WEIGHT_RECORDS.append(weight_frame)
        print(
            "Ensemble complete:",
            outer["OuterFoldID"],
            "members=",
            len(weight_frame),
        )
    except Exception as exc:
        failure_record(
            outer=outer,
            model_name="constrained_ensemble",
            exc=exc,
        )
        print(
            "ENSEMBLE FAILED:",
            outer["OuterFoldID"],
            repr(exc),
        )
    finally:
        gc.collect()

if ENSEMBLE_WEIGHT_RECORDS:
    ensemble_weights = pd.concat(
        ENSEMBLE_WEIGHT_RECORDS, ignore_index=True
    )
    ensemble_weights.to_csv(
        MODEL_REPORTS / f"ensemble_weights_{RUN_MODE}.csv",
        index=False,
    )
else:
    ensemble_weights = pd.DataFrame()

print("Completed ensembles:", len(ENSEMBLE_PREDICTIONS))


## 12. Prequential partial pooling

The pooled common-feature model may reduce variance, while gender-specific systems can exploit different histories and men-only sources. The partial-pooling challenger combines them without using the current validation season to choose its weight.

For validation season `Y`, the gender-specific blend weight is estimated only from aligned outer out-of-fold predictions from seasons `< Y`. When too little prior evidence exists, a conservative pre-registered weight is used.


In [ ]:
def optimize_two_model_weight(
    frame: pd.DataFrame,
    *,
    separate_column: str,
    pooled_column: str,
) -> float:
    y = frame["Team1Win"].to_numpy(dtype=float)
    separate = clip_probability(frame[separate_column])
    pooled = clip_probability(frame[pooled_column])
    seasons = frame["Season"].to_numpy(dtype=int)

    def objective(weight: float) -> float:
        prediction = weight * separate + (1.0 - weight) * pooled
        return macro_season_brier_from_arrays(
            y,
            prediction,
            seasons,
        ) + 0.001 * (weight - 0.5) ** 2

    result = minimize_scalar(
        objective,
        bounds=(0.0, 1.0),
        method="bounded",
    )
    return float(np.clip(result.x, 0.0, 1.0))


def ensemble_outer_table() -> pd.DataFrame:
    if not ENSEMBLE_PREDICTIONS:
        return pd.DataFrame()
    return pd.concat(
        ENSEMBLE_PREDICTIONS, ignore_index=True
    )


ensemble_outer_all = ensemble_outer_table()
PARTIAL_POOLING_PREDICTIONS: list[pd.DataFrame] = []
PARTIAL_POOLING_WEIGHTS: list[dict[str, Any]] = []

if not ensemble_outer_all.empty:
    rich_ensemble = ensemble_outer_all.loc[
        ensemble_outer_all["Universe"].eq("rich")
    ].copy()

    aligned_history_by_gender: dict[str, pd.DataFrame] = {}
    for gender in ("M", "W"):
        separate = rich_ensemble.loc[
            rich_ensemble["Architecture"].eq("separate_gender")
            & rich_ensemble["Gender"].eq(gender)
        ][
            [
                "TargetKey",
                "Gender",
                "Season",
                "Team1ID",
                "Team2ID",
                "Team1Win",
                "Team1Margin",
                "Prediction",
            ]
        ].rename(columns={"Prediction": "SeparatePrediction"})

        pooled = rich_ensemble.loc[
            rich_ensemble["Architecture"].eq("pooled_common")
            & rich_ensemble["Gender"].eq(gender)
        ][["TargetKey", "Prediction"]].rename(
            columns={"Prediction": "PooledPrediction"}
        )

        aligned = separate.merge(
            pooled,
            on="TargetKey",
            how="inner",
            validate="one_to_one",
        ).sort_values(["Season", "TargetKey"])
        aligned_history_by_gender[gender] = aligned

        for validation_season in sorted(
            map(int, aligned["Season"].unique())
        ):
            prior = aligned.loc[
                aligned["Season"].lt(validation_season)
            ]
            if (
                prior["Season"].nunique() >= 2
                and len(prior) >= 100
            ):
                weight = optimize_two_model_weight(
                    prior,
                    separate_column="SeparatePrediction",
                    pooled_column="PooledPrediction",
                )
                source = "prior_aligned_outer_oof"
            else:
                weight = 0.75
                source = "preregistered_cold_start"

            current = aligned.loc[
                aligned["Season"].eq(validation_season)
            ].copy()
            current["Prediction"] = clip_probability(
                weight * current["SeparatePrediction"]
                + (1.0 - weight) * current["PooledPrediction"]
            )
            current["RawPrediction"] = current["Prediction"]
            current["OuterFoldID"] = (
                f"rich_partial_{gender}_{validation_season}"
            )
            current["InnerFoldID"] = None
            current["Architecture"] = "partial_pooling"
            current["Universe"] = "rich"
            current["Model"] = "partial_pooling_ensemble"
            current["CalibrationMethod"] = (
                "component_calibrated_linear_pool"
            )
            current["FeatureCount"] = np.nan
            current["BestIteration"] = np.nan
            PARTIAL_POOLING_PREDICTIONS.append(
                current[
                    [
                        "TargetKey",
                        "Gender",
                        "Season",
                        "Team1ID",
                        "Team2ID",
                        "Team1Win",
                        "Team1Margin",
                        "OuterFoldID",
                        "InnerFoldID",
                        "Architecture",
                        "Universe",
                        "Model",
                        "RawPrediction",
                        "Prediction",
                        "CalibrationMethod",
                        "FeatureCount",
                        "BestIteration",
                    ]
                ]
            )
            PARTIAL_POOLING_WEIGHTS.append(
                {
                    "Gender": gender,
                    "ValidationSeason": validation_season,
                    "SeparateWeight": weight,
                    "PooledWeight": 1.0 - weight,
                    "WeightSource": source,
                    "PriorSeasons": sorted(
                        map(int, prior["Season"].unique())
                    ),
                    "PriorRows": len(prior),
                }
            )

partial_pooling_weights = pd.DataFrame(
    PARTIAL_POOLING_WEIGHTS
)
partial_pooling_weights.to_csv(
    MODEL_REPORTS / f"partial_pooling_weights_{RUN_MODE}.csv",
    index=False,
)

print(
    "Partial-pooling prediction blocks:",
    len(PARTIAL_POOLING_PREDICTIONS),
)
partial_pooling_weights.tail(10)


## 13. Development OOF leaderboard and calibration diagnostics

All model comparisons below use development outer-fold predictions only.

The report distinguishes:

- compact versus rich feature universes;
- separate versus pooled versus partial-pooling architectures;
- men, women, and all-row summaries;
- macro season Brier versus game-weighted Brier;
- calibrated versus raw prediction behavior.

A model must improve more than the pooled mean alone: it should remain stable across seasons and genders.


In [ ]:
prediction_parts = list(OUTER_PREDICTIONS) + list(
    ENSEMBLE_PREDICTIONS
) + list(PARTIAL_POOLING_PREDICTIONS)
assert prediction_parts, "No development OOF predictions were generated."

development_oof = pd.concat(
    prediction_parts,
    ignore_index=True,
)
development_oof["Prediction"] = clip_probability(
    development_oof["Prediction"]
)
assert development_oof["Season"].max() <= DEVELOPMENT_LAST_SEASON
assert not development_oof["Season"].isin(LOCKED_SEASONS).any()
assert development_oof["Team1Win"].isin([0, 1]).all()
assert development_oof["Prediction"].between(0, 1).all()

duplicate_keys = development_oof.duplicated(
    ["OuterFoldID", "Model", "TargetKey"]
).sum()
assert duplicate_keys == 0, (
    f"Duplicate OOF predictions: {duplicate_keys}"
)

oof_path = (
    PROCESSED
    / f"development_oof_predictions_v1_{RUN_MODE}.parquet"
)
atomic_write_parquet(oof_path, development_oof)

METRIC_RECORDS: list[dict[str, Any]] = []
SEASON_METRIC_RECORDS: list[pd.DataFrame] = []


def add_metric_group(
    group: pd.DataFrame,
    *,
    architecture: str,
    universe: str,
    model: str,
    evaluation_gender: str,
) -> None:
    metrics, by_season = evaluate_probability_frame(group)
    METRIC_RECORDS.append(
        {
            "Architecture": architecture,
            "Universe": universe,
            "Model": model,
            "EvaluationGender": evaluation_gender,
            "ComplexityRank": (
                MODEL_SPECS[model].complexity_rank
                if model in MODEL_SPECS
                else 11
                if model == "constrained_ensemble"
                else 12
            ),
            **metrics,
        }
    )
    by_season["Architecture"] = architecture
    by_season["Universe"] = universe
    by_season["Model"] = model
    by_season["EvaluationGender"] = evaluation_gender
    SEASON_METRIC_RECORDS.append(by_season)


for (
    architecture,
    universe,
    model,
), group in development_oof.groupby(
    ["Architecture", "Universe", "Model"],
    observed=True,
):
    add_metric_group(
        group,
        architecture=str(architecture),
        universe=str(universe),
        model=str(model),
        evaluation_gender="ALL",
    )
    for gender, gender_group in group.groupby(
        "Gender", observed=True
    ):
        add_metric_group(
            gender_group,
            architecture=str(architecture),
            universe=str(universe),
            model=str(model),
            evaluation_gender=str(gender),
        )

leaderboard = pd.DataFrame(METRIC_RECORDS).sort_values(
    [
        "EvaluationGender",
        "Universe",
        "MacroSeasonBrier",
        "ComplexityRank",
    ]
).reset_index(drop=True)
season_metrics = pd.concat(
    SEASON_METRIC_RECORDS,
    ignore_index=True,
)

leaderboard.to_csv(
    MODEL_REPORTS / f"development_leaderboard_{RUN_MODE}.csv",
    index=False,
)
season_metrics.to_csv(
    MODEL_REPORTS
    / f"development_brier_by_season_{RUN_MODE}.csv",
    index=False,
)

display_columns = [
    "Architecture",
    "Universe",
    "Model",
    "EvaluationGender",
    "Rows",
    "Seasons",
    "MacroSeasonBrier",
    "GameWeightedBrier",
    "SeasonBrierStd",
    "WorstSeasonBrier",
    "LogLoss",
    "ROCAUC",
    "CalibrationIntercept",
    "CalibrationSlope",
    "ECE",
]
leaderboard[display_columns].head(40)


In [ ]:
# Common rich comparison window for men/women/pooled/partial pooling.
rich_common = development_oof.loc[
    development_oof["Universe"].eq("rich")
].copy()
if not rich_common.empty:
    common_seasons_by_gender = {}
    for gender in ("M", "W"):
        model_season_sets = [
            set(map(int, group["Season"].unique()))
            for _, group in rich_common.loc[
                rich_common["Gender"].eq(gender)
            ].groupby(
                ["Architecture", "Model"], observed=True
            )
            if len(group)
        ]
        common_seasons_by_gender[gender] = (
            sorted(set.intersection(*model_season_sets))
            if model_season_sets
            else []
        )

    common_records = []
    for gender, common_seasons in common_seasons_by_gender.items():
        for (
            architecture,
            model,
        ), group in rich_common.loc[
            rich_common["Gender"].eq(gender)
            & rich_common["Season"].isin(common_seasons)
        ].groupby(["Architecture", "Model"], observed=True):
            if not common_seasons:
                continue
            metrics, _ = evaluate_probability_frame(group)
            common_records.append(
                {
                    "Gender": gender,
                    "Architecture": architecture,
                    "Model": model,
                    "CommonSeasons": json.dumps(common_seasons),
                    **metrics,
                }
            )
    common_window_leaderboard = pd.DataFrame(
        common_records
    ).sort_values(["Gender", "MacroSeasonBrier"])
else:
    common_window_leaderboard = pd.DataFrame()

common_window_leaderboard.to_csv(
    MODEL_REPORTS
    / f"common_window_architecture_comparison_{RUN_MODE}.csv",
    index=False,
)
common_window_leaderboard.head(30)


In [ ]:
# Reliability curves and season Brier heatmaps.
def reliability_table(
    frame: pd.DataFrame,
    *,
    bins: int = 10,
) -> pd.DataFrame:
    p = clip_probability(frame["Prediction"])
    try:
        labels = pd.qcut(
            pd.Series(p),
            q=min(bins, max(2, len(np.unique(p)))),
            duplicates="drop",
        )
    except Exception:
        labels = pd.cut(
            pd.Series(p),
            bins=min(bins, max(2, len(np.unique(p)))),
            duplicates="drop",
        )
    output = pd.DataFrame(
        {
            "Prediction": p,
            "Outcome": frame["Team1Win"].to_numpy(dtype=float),
            "Bin": labels,
        }
    )
    return (
        output.groupby("Bin", observed=True)
        .agg(
            Rows=("Outcome", "size"),
            MeanPrediction=("Prediction", "mean"),
            ObservedRate=("Outcome", "mean"),
        )
        .reset_index(drop=True)
    )


top_models_to_plot: list[tuple[str, str, str]] = []
for gender in ("M", "W"):
    candidates = leaderboard.loc[
        leaderboard["EvaluationGender"].eq(gender)
        & leaderboard["Universe"].eq("rich")
    ]
    if not candidates.empty:
        for _, row in candidates.head(4).iterrows():
            top_models_to_plot.append(
                (
                    str(row["Architecture"]),
                    str(row["Model"]),
                    gender,
                )
            )

for gender in ("M", "W"):
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    plotted = 0
    for architecture, model, model_gender in top_models_to_plot:
        if model_gender != gender:
            continue
        subset = development_oof.loc[
            development_oof["Gender"].eq(gender)
            & development_oof["Architecture"].eq(architecture)
            & development_oof["Model"].eq(model)
            & development_oof["Universe"].eq("rich")
        ]
        if subset.empty:
            continue
        curve = reliability_table(subset)
        ax.plot(
            curve["MeanPrediction"],
            curve["ObservedRate"],
            marker="o",
            label=f"{architecture}/{model}",
        )
        plotted += 1
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed Team1 win rate")
    ax.set_title(f"{gender}: development OOF reliability")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / f"development_reliability_{gender}_{RUN_MODE}.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()

# Heatmap for the top rich models by common-window performance.
for gender in ("M", "W"):
    ranking = common_window_leaderboard.loc[
        common_window_leaderboard["Gender"].eq(gender)
    ].head(10)
    if ranking.empty:
        continue
    keys = list(zip(ranking["Architecture"], ranking["Model"]))
    table = season_metrics.loc[
        season_metrics["EvaluationGender"].eq(gender)
        & season_metrics["Universe"].eq("rich")
    ].copy()
    table["ArchitectureModel"] = (
        table["Architecture"].astype(str)
        + " / "
        + table["Model"].astype(str)
    )
    labels = [f"{a} / {m}" for a, m in keys]
    pivot = (
        table.loc[table["ArchitectureModel"].isin(labels)]
        .pivot_table(
            index="ArchitectureModel",
            columns="Season",
            values="Brier",
            aggfunc="mean",
        )
        .reindex(labels)
    )
    if pivot.empty:
        continue
    fig, ax = plt.subplots(
        figsize=(max(10, pivot.shape[1] * 0.7), max(5, pivot.shape[0] * 0.55))
    )
    image = ax.imshow(pivot.to_numpy(), aspect="auto")
    ax.set_xticks(
        np.arange(pivot.shape[1]),
        labels=list(map(str, pivot.columns)),
        rotation=45,
    )
    ax.set_yticks(
        np.arange(pivot.shape[0]),
        labels=pivot.index,
    )
    ax.set_title(f"{gender}: Brier score by held-out tournament season")
    fig.colorbar(image, ax=ax, label="Brier score")
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / f"season_brier_heatmap_{gender}_{RUN_MODE}.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


## 14. Pre-registered feature-block ablation

A broad feature bank is not evidence that every feature family helps. This section tests cumulative basketball feature blocks with a fixed model recipe under the same season-held-out protocol.

To remain practical:

- `standard` mode uses elastic-net logistic regression across every rich outer fold;
- `exhaustive` mode adds XGBoost and leave-one-block-out experiments;
- `smoke` mode skips ablation.

The block experiments are development analyses. They cannot inspect the locked benchmark.


In [ ]:
ABLATION_LADDER: list[dict[str, Any]] = [
    {
        "Stage": 0,
        "Experiment": "constant_probability",
        "Blocks": [],
    },
    {
        "Stage": 1,
        "Experiment": "seed_only",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
        ],
    },
    {
        "Stage": 2,
        "Experiment": "compact_basic",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
        ],
    },
    {
        "Stage": 3,
        "Experiment": "compact_plus_ratings",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
        ],
    },
    {
        "Stage": 4,
        "Experiment": "schedule_context",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
        ],
    },
    {
        "Stage": 5,
        "Experiment": "rich_efficiency",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
            "detailed_efficiency",
            "opponent_adjusted_efficiency",
        ],
    },
    {
        "Stage": 6,
        "Experiment": "history_context",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
            "detailed_efficiency",
            "opponent_adjusted_efficiency",
            "prior_program_history",
            "prior_coach_history",
        ],
    },
    {
        "Stage": 7,
        "Experiment": "men_massey",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
            "detailed_efficiency",
            "opponent_adjusted_efficiency",
            "prior_program_history",
            "prior_coach_history",
            "massey_consensus",
        ],
    },
    {
        "Stage": 8,
        "Experiment": "matchup_interactions",
        "Blocks": [
            "selection_committee_prior",
            "prequential_seed_priors",
            "compact_performance",
            "availability_and_context",
            "dynamic_ratings",
            "global_strength_ratings",
            "within_season_normalization",
            "schedule_and_accomplishment",
            "conference_context",
            "detailed_efficiency",
            "opponent_adjusted_efficiency",
            "prior_program_history",
            "prior_coach_history",
            "massey_consensus",
            "matchup_interactions",
        ],
    },
    {
        "Stage": 9,
        "Experiment": "full_candidate_bank",
        "Blocks": sorted(set(FEATURE_TO_BLOCK.values())),
    },
]


def candidates_for_blocks(
    candidates: Sequence[str],
    blocks: Sequence[str],
) -> list[str]:
    block_set = set(blocks)
    return [
        feature
        for feature in candidates
        if FEATURE_TO_BLOCK.get(feature, "other") in block_set
    ]


def run_ablation_fold(
    outer: pd.Series,
    experiment: dict[str, Any],
    *,
    model_name: str,
) -> pd.DataFrame:
    outer_fold_id = str(outer["OuterFoldID"])
    experiment_name = str(experiment["Experiment"])
    cache_name = f"ablation__{experiment_name}__{model_name}"
    directory = outer_model_directory(outer_fold_id, cache_name)
    if checkpoint_is_valid(directory):
        return pd.read_parquet(
            directory / "outer_predictions.parquet"
        )

    gender = str(outer["Gender"])
    universe = str(outer["Universe"])
    pooled = gender == "Pooled"
    train_seasons = parse_json_int_list(
        outer["TrainingSeasonsJSON"]
    )
    validation_season = int(outer["ValidationSeason"])
    training = rows_for_context(
        development,
        seasons=train_seasons,
        gender=gender,
        universe=universe,
    )
    validation = rows_for_context(
        development,
        seasons=[validation_season],
        gender=gender,
        universe=universe,
    )

    if experiment_name == "constant_probability":
        probability = np.full(len(validation), 0.5)
        output = make_prediction_frame(
            validation,
            outer_fold_id=outer_fold_id,
            inner_fold_id=None,
            architecture=str(outer["Architecture"]),
            universe=universe,
            model_name=cache_name,
            raw_prediction=probability,
            feature_count=0,
            best_iteration=None,
        )
        output["Prediction"] = probability
        output["CalibrationMethod"] = "identity"
    else:
        all_candidates = candidates_for_context(
            gender=gender,
            universe=universe,
            seed_aware=True,
        )
        block_candidates = candidates_for_blocks(
            all_candidates,
            experiment["Blocks"],
        )
        assert block_candidates, (
            f"No candidates for ablation {experiment_name}"
        )
        spec = MODEL_SPECS[model_name]
        params = default_parameters(spec)

        # Build inner OOF manually with the block-restricted bank.
        inner_oof, best_iterations, _ = run_inner_oof(
            outer=outer,
            spec=spec,
            params=params,
            candidates=block_candidates,
            collect_selector_audits=False,
        )
        calibration = select_cross_fitted_calibrator(
            inner_oof,
            raw_kind=spec.raw_kind,
        )
        features, selection = model_feature_list(
            spec,
            training,
            block_candidates,
            pooled=pooled,
            selection_key=(
                f"{outer_fold_id}__ablation__{experiment_name}"
                f"__{model_name}"
            ),
        )
        fixed_iterations = (
            max(10, int(round(float(np.median(best_iterations)))))
            if best_iterations
            else None
        )
        fitted, raw = fit_predictor(
            spec,
            params,
            training,
            validation,
            features,
            fixed_iterations=fixed_iterations,
            random_seed=SEED + validation_season,
        )
        output = make_prediction_frame(
            validation,
            outer_fold_id=outer_fold_id,
            inner_fold_id=None,
            architecture=str(outer["Architecture"]),
            universe=universe,
            model_name=cache_name,
            raw_prediction=raw,
            feature_count=len(features),
            best_iteration=fitted.best_iteration,
        )
        output["Prediction"] = (
            calibration.fitted_calibrator.predict(raw)
        )
        output["CalibrationMethod"] = calibration.method

    metadata = {
        "status": "complete",
        "contracts": EXPECTED_CONTRACTS,
        "run_mode": RUN_MODE,
        "outer_fold_id": outer_fold_id,
        "model": cache_name,
        "outer_feature_count": int(output["FeatureCount"].iloc[0]),
        "outer_metrics": evaluate_probability_frame(output)[0],
        "calibration_method": str(output["CalibrationMethod"].iloc[0]),
    }
    directory.mkdir(parents=True, exist_ok=True)
    atomic_write_parquet(
        directory / "outer_predictions.parquet", output
    )
    # A placeholder inner artifact keeps checkpoint validation uniform.
    atomic_write_parquet(
        directory / "inner_oof.parquet",
        output.iloc[0:0].copy(),
    )
    atomic_write_json(directory / "metadata.json", metadata)
    return output


ABLATION_PREDICTIONS: list[pd.DataFrame] = []
if MODE["run_ablation"]:
    ablation_models = (
        ["elastic_logistic", "xgb_classifier"]
        if RUN_MODE == "exhaustive"
        else ["elastic_logistic"]
    )
    rich_outer_plan = outer_plan.loc[
        outer_plan["Universe"].eq("rich")
    ].copy()

    for _, outer in rich_outer_plan.iterrows():
        for experiment in ABLATION_LADDER:
            for model_name in ablation_models:
                try:
                    print(
                        "Ablation:",
                        outer["OuterFoldID"],
                        experiment["Experiment"],
                        model_name,
                    )
                    ensure_memory_headroom(
                        f"ablation {outer['OuterFoldID']} "
                        f"{experiment['Experiment']} {model_name}"
                    )
                    result = run_ablation_fold(
                        outer,
                        experiment,
                        model_name=model_name,
                    )
                    result["AblationExperiment"] = experiment[
                        "Experiment"
                    ]
                    result["AblationStage"] = int(
                        experiment["Stage"]
                    )
                    result["AblationBaseModel"] = model_name
                    ABLATION_PREDICTIONS.append(result)
                except Exception as exc:
                    failure_record(
                        outer=outer,
                        model_name=(
                            f"ablation__{experiment['Experiment']}"
                            f"__{model_name}"
                        ),
                        exc=exc,
                    )
                    print("ABLATION FAILED:", repr(exc))
                finally:
                    gc.collect()

if ABLATION_PREDICTIONS:
    ablation_oof = pd.concat(
        ABLATION_PREDICTIONS, ignore_index=True
    )
    atomic_write_parquet(
        PROCESSED
        / f"development_feature_ablation_oof_{RUN_MODE}.parquet",
        ablation_oof,
    )
    ablation_records = []
    for (
        experiment,
        stage,
        base_model,
        gender,
        architecture,
    ), group in ablation_oof.groupby(
        [
            "AblationExperiment",
            "AblationStage",
            "AblationBaseModel",
            "Gender",
            "Architecture",
        ],
        observed=True,
    ):
        metrics, _ = evaluate_probability_frame(group)
        ablation_records.append(
            {
                "Experiment": experiment,
                "Stage": int(stage),
                "BaseModel": base_model,
                "Gender": gender,
                "Architecture": architecture,
                **metrics,
            }
        )
    ablation_leaderboard = pd.DataFrame(
        ablation_records
    ).sort_values(
        ["Gender", "Architecture", "BaseModel", "Stage"]
    )
else:
    ablation_oof = pd.DataFrame()
    ablation_leaderboard = pd.DataFrame()

ablation_leaderboard.to_csv(
    MODEL_REPORTS / f"feature_block_ablation_{RUN_MODE}.csv",
    index=False,
)
ablation_leaderboard.head(30)


## 15. Dimensionality, feature-stability, runtime, and failure diagnostics

The notebook treats feature selection itself as an object to validate.

Reports include:

- selected feature count by fold and model;
- selection frequency by feature and block;
- Jaccard overlap across consecutive outer folds;
- mandatory-signal retention;
- feature-count compliance with adaptive caps;
- model elapsed time and RSS change;
- all model or ablation failures with tracebacks.

A feature that appears in only one favorable season is less trustworthy than one selected repeatedly across historical deployments.


In [ ]:
# Reconstruct selection records from disk so resumed runs are included.
selection_files = list(
    (CACHE_DIR / "outer_models").glob("*/*/selected_features.csv")
)
selection_frames = []
for path in selection_files:
    if "ablation__" in str(path.parent.name):
        continue
    try:
        frame = pd.read_csv(path)
        frame["SelectionFile"] = str(path)
        selection_frames.append(frame)
    except Exception:
        continue

selected_feature_audit = (
    pd.concat(selection_frames, ignore_index=True)
    if selection_frames
    else pd.DataFrame()
)

if not selected_feature_audit.empty:
    selected_only = selected_feature_audit.loc[
        selected_feature_audit["Selected"].astype(bool)
    ].copy()
    selected_only["Block"] = selected_only["Feature"].map(
        FEATURE_TO_BLOCK
    ).fillna(selected_only.get("Block", "other"))
    feature_stability = (
        selected_only.groupby(
            ["Model", "Feature", "Block"], observed=True
        )
        .agg(
            SelectionOccurrences=("OuterFoldID", "nunique"),
            MeanStabilityScore=("StabilityScore", "mean"),
            MedianSelectionRank=("SelectionRank", "median"),
            MeanMissingRate=("MissingRate", "mean"),
        )
        .reset_index()
    )
    model_fold_counts = (
        selected_only.groupby("Model", observed=True)[
            "OuterFoldID"
        ].nunique()
    )
    feature_stability["EligibleOuterFolds"] = (
        feature_stability["Model"].map(model_fold_counts)
    )
    feature_stability["SelectionFrequency"] = (
        feature_stability["SelectionOccurrences"]
        / feature_stability["EligibleOuterFolds"]
    )
    block_stability = (
        selected_only.groupby(["Model", "Block"], observed=True)
        .agg(
            SelectedFeatureOccurrences=("Feature", "size"),
            UniqueFeatures=("Feature", "nunique"),
            OuterFolds=("OuterFoldID", "nunique"),
        )
        .reset_index()
    )

    jaccard_records = []
    for model, model_rows in selected_only.groupby(
        "Model", observed=True
    ):
        feature_sets = {
            fold: set(group["Feature"])
            for fold, group in model_rows.groupby(
                "OuterFoldID", observed=True
            )
        }
        fold_order = (
            outer_plan.loc[
                outer_plan["OuterFoldID"].isin(feature_sets)
            ]
            .sort_values(
                ["Gender", "Architecture", "Universe", "ValidationSeason"]
            )
        )
        for (
            gender,
            architecture,
            universe,
        ), context in fold_order.groupby(
            ["Gender", "Architecture", "Universe"],
            observed=True,
        ):
            ordered_folds = context["OuterFoldID"].tolist()
            for previous, current in zip(
                ordered_folds[:-1],
                ordered_folds[1:],
                strict=True,
            ):
                a = feature_sets[previous]
                b = feature_sets[current]
                union = a | b
                jaccard_records.append(
                    {
                        "Model": model,
                        "Gender": gender,
                        "Architecture": architecture,
                        "Universe": universe,
                        "PreviousFold": previous,
                        "CurrentFold": current,
                        "Jaccard": (
                            len(a & b) / len(union)
                            if union
                            else np.nan
                        ),
                        "PreviousFeatures": len(a),
                        "CurrentFeatures": len(b),
                    }
                )
    selection_jaccard = pd.DataFrame(jaccard_records)
else:
    feature_stability = pd.DataFrame()
    block_stability = pd.DataFrame()
    selection_jaccard = pd.DataFrame()

selected_feature_audit.to_csv(
    MODEL_REPORTS / f"selected_feature_audit_{RUN_MODE}.csv",
    index=False,
)
feature_stability.to_csv(
    MODEL_REPORTS / f"feature_selection_stability_{RUN_MODE}.csv",
    index=False,
)
block_stability.to_csv(
    MODEL_REPORTS / f"selected_feature_blocks_{RUN_MODE}.csv",
    index=False,
)
selection_jaccard.to_csv(
    MODEL_REPORTS / f"feature_selection_jaccard_{RUN_MODE}.csv",
    index=False,
)

feature_count_audit = (
    development_oof.loc[
        development_oof["FeatureCount"].notna()
    ]
    .groupby(
        ["Architecture", "Universe", "Model", "Gender"],
        observed=True,
    )["FeatureCount"]
    .agg(["min", "median", "max"])
    .reset_index()
)
feature_count_audit.to_csv(
    MODEL_REPORTS / f"feature_count_audit_{RUN_MODE}.csv",
    index=False,
)

# Standardized linear coefficients are aggregated across rolling folds.
# This diagnoses sign instability and one-season coefficient flips that a
# single fitted model would conceal.
coefficient_files = [
    path
    for path in (CACHE_DIR / "outer_models").glob(
        "*/*/linear_coefficients.csv"
    )
    if "ablation__" not in path.parent.name
]
coefficient_frames = []
for path in coefficient_files:
    try:
        frame = pd.read_csv(path)
        frame["CoefficientFile"] = str(path)
        coefficient_frames.append(frame)
    except Exception:
        continue

linear_coefficients = (
    pd.concat(coefficient_frames, ignore_index=True)
    if coefficient_frames
    else pd.DataFrame()
)
if not linear_coefficients.empty:
    linear_coefficients["CoefficientSign"] = np.sign(
        linear_coefficients["StandardizedCoefficient"]
    )
    coefficient_stability = (
        linear_coefficients.groupby(
            [
                "Gender",
                "Architecture",
                "Universe",
                "Model",
                "Feature",
                "Block",
            ],
            observed=True,
        )
        .agg(
            OuterFolds=("OuterFoldID", "nunique"),
            MeanCoefficient=("StandardizedCoefficient", "mean"),
            MedianCoefficient=("StandardizedCoefficient", "median"),
            MeanAbsoluteCoefficient=(
                "StandardizedCoefficient",
                lambda values: float(np.mean(np.abs(values))),
            ),
            CoefficientStd=("StandardizedCoefficient", "std"),
            PositiveRate=(
                "StandardizedCoefficient",
                lambda values: float(np.mean(np.asarray(values) > 0)),
            ),
            NegativeRate=(
                "StandardizedCoefficient",
                lambda values: float(np.mean(np.asarray(values) < 0)),
            ),
        )
        .reset_index()
    )
    coefficient_stability["SignConsistency"] = np.maximum(
        coefficient_stability["PositiveRate"],
        coefficient_stability["NegativeRate"],
    )
    coefficient_stability = coefficient_stability.sort_values(
        [
            "Gender",
            "Architecture",
            "Universe",
            "Model",
            "MeanAbsoluteCoefficient",
        ],
        ascending=[True, True, True, True, False],
    )
    coefficient_block_stability = (
        coefficient_stability.groupby(
            ["Gender", "Architecture", "Universe", "Model", "Block"],
            observed=True,
        )
        .agg(
            UniqueFeatures=("Feature", "nunique"),
            MedianSignConsistency=("SignConsistency", "median"),
            MeanAbsoluteCoefficient=("MeanAbsoluteCoefficient", "mean"),
        )
        .reset_index()
    )
else:
    coefficient_stability = pd.DataFrame()
    coefficient_block_stability = pd.DataFrame()

linear_coefficients.to_csv(
    MODEL_REPORTS / f"linear_coefficients_all_{RUN_MODE}.csv",
    index=False,
)
coefficient_stability.to_csv(
    MODEL_REPORTS / f"linear_coefficient_stability_{RUN_MODE}.csv",
    index=False,
)
coefficient_block_stability.to_csv(
    MODEL_REPORTS / f"linear_coefficient_block_stability_{RUN_MODE}.csv",
    index=False,
)

resource_log = pd.DataFrame(RESOURCE_LOG)
resource_log.to_csv(
    MODEL_REPORTS / f"resource_usage_{RUN_MODE}.csv",
    index=False,
)
failure_frame = pd.DataFrame(MODEL_FAILURES)
failure_frame.to_csv(
    MODEL_REPORTS / f"all_failures_{RUN_MODE}.csv",
    index=False,
)

print("Selection files:", len(selection_files))
print("Linear coefficient files:", len(coefficient_files))
print("Model failures:", len(failure_frame))
display(feature_count_audit.head(30))
if not selection_jaccard.empty:
    display(
        selection_jaccard.groupby("Model", observed=True)["Jaccard"]
        .agg(["count", "mean", "median", "min"])
        .reset_index()
    )


## 16. Error slices and paired season-clustered comparisons

The same average Brier score can hide very different failure modes. This section examines:

- confident wrong predictions;
- seed-gap bands;
- tournament-day bands;
- gender;
- season;
- paired loss differences between the selected candidate and transparent baselines.

The bootstrap resamples whole seasons, preserving tournament-level dependence better than treating every game as independent.


In [ ]:
development_metadata = development[
    [
        column
        for column in (
            "TargetKey",
            "DayNum",
            "Team1ID",
            "Team2ID",
            "matchup__seed_gap_abs",
            "matchup__seed_diff",
        )
        if column in development.columns
    ]
].drop_duplicates("TargetKey")

oof_with_context = development_oof.merge(
    development_metadata,
    on=[
        column
        for column in ["TargetKey"]
        if column in development_metadata.columns
    ],
    how="left",
    validate="many_to_one",
    suffixes=("", "_source"),
)
oof_with_context["SquaredError"] = (
    oof_with_context["Prediction"]
    - oof_with_context["Team1Win"]
) ** 2
oof_with_context["Confidence"] = np.maximum(
    oof_with_context["Prediction"],
    1.0 - oof_with_context["Prediction"],
)
oof_with_context["WrongAtHalf"] = (
    (oof_with_context["Prediction"] >= 0.5).astype(int)
    != oof_with_context["Team1Win"].astype(int)
)
oof_with_context["ConfidentWrong"] = (
    oof_with_context["WrongAtHalf"]
    & oof_with_context["Confidence"].ge(0.80)
)

if "matchup__seed_gap_abs" in oof_with_context.columns:
    oof_with_context["SeedGapBand"] = pd.cut(
        oof_with_context["matchup__seed_gap_abs"],
        bins=[-np.inf, 0, 2, 5, 8, 12, np.inf],
        labels=["0", "1-2", "3-5", "6-8", "9-12", "13+"],
    )
else:
    oof_with_context["SeedGapBand"] = "unavailable"

if "DayNum" in oof_with_context.columns:
    oof_with_context["TournamentDayBand"] = pd.cut(
        oof_with_context["DayNum"],
        bins=[-np.inf, 135, 139, 146, 152, np.inf],
        labels=[
            "play_in",
            "rounds_1_2",
            "rounds_3_4",
            "semifinal",
            "final_or_other",
        ],
    )
else:
    oof_with_context["TournamentDayBand"] = "unavailable"

error_slice_records = []
for (
    architecture,
    universe,
    model,
    gender,
    seed_band,
), group in oof_with_context.groupby(
    [
        "Architecture",
        "Universe",
        "Model",
        "Gender",
        "SeedGapBand",
    ],
    observed=True,
):
    error_slice_records.append(
        {
            "Architecture": architecture,
            "Universe": universe,
            "Model": model,
            "Gender": gender,
            "SeedGapBand": str(seed_band),
            "Rows": len(group),
            "Brier": float(group["SquaredError"].mean()),
            "ConfidentWrongRate": float(
                group["ConfidentWrong"].mean()
            ),
            "MeanConfidence": float(group["Confidence"].mean()),
        }
    )
error_slices = pd.DataFrame(error_slice_records)
error_slices.to_csv(
    MODEL_REPORTS / f"error_slices_seed_gap_{RUN_MODE}.csv",
    index=False,
)

round_slice_records = []
for (
    architecture,
    universe,
    model,
    gender,
    round_band,
), group in oof_with_context.groupby(
    [
        "Architecture",
        "Universe",
        "Model",
        "Gender",
        "TournamentDayBand",
    ],
    observed=True,
):
    round_slice_records.append(
        {
            "Architecture": architecture,
            "Universe": universe,
            "Model": model,
            "Gender": gender,
            "TournamentDayBand": str(round_band),
            "Rows": len(group),
            "Brier": float(group["SquaredError"].mean()),
            "ConfidentWrongRate": float(
                group["ConfidentWrong"].mean()
            ),
            "MeanConfidence": float(group["Confidence"].mean()),
        }
    )
round_error_slices = pd.DataFrame(round_slice_records)
round_error_slices.to_csv(
    MODEL_REPORTS / f"error_slices_tournament_round_{RUN_MODE}.csv",
    index=False,
)

worst_errors = (
    oof_with_context.sort_values(
        ["SquaredError", "Confidence"],
        ascending=False,
    )
    .head(250)
)
worst_errors.to_csv(
    MODEL_REPORTS / f"largest_oof_errors_{RUN_MODE}.csv",
    index=False,
)

print(
    "Confident wrong predictions:",
    int(oof_with_context["ConfidentWrong"].sum()),
)
worst_errors[
    [
        "Season",
        "Gender",
        "Architecture",
        "Model",
        "Team1ID",
        "Team2ID",
        "Team1Win",
        "Prediction",
        "SquaredError",
    ]
].head(20)


## 17. Freeze a development-selected recipe without opening the benchmark

The recipe is selected separately for men and women on a **matched rich-era season window**.

The one-standard-error rule prevents choosing a much more complicated architecture for a numerically tiny development advantage. The recipe stores:

- chosen prediction architecture;
- model or ensemble name;
- dimensionality controls;
- search and calibration procedures;
- fallback seed-free strategy;
- all source fingerprints.

It stores no locked-benchmark result because those outcomes have not been loaded.


In [ ]:
def stream_complexity(architecture: str, model: str) -> int:
    if model in MODEL_SPECS:
        return MODEL_SPECS[model].complexity_rank
    if model == "constrained_ensemble":
        return 11
    if model == "partial_pooling_ensemble":
        return 12
    return 99


def select_stream_one_se(
    frame: pd.DataFrame,
    *,
    gender: str,
) -> tuple[dict[str, Any], pd.DataFrame]:
    subset = frame.loc[
        frame["Gender"].eq(gender)
        & frame["Universe"].eq("rich")
    ].copy()
    assert not subset.empty

    # Matched era: require the pooled challenger era when available.
    pooled_seasons = set(
        map(
            int,
            subset.loc[
                subset["Architecture"].eq("pooled_common"),
                "Season",
            ].unique(),
        )
    )
    separate_seasons = set(
        map(
            int,
            subset.loc[
                subset["Architecture"].eq("separate_gender"),
                "Season",
            ].unique(),
        )
    )
    comparison_seasons = sorted(
        pooled_seasons.intersection(separate_seasons)
    )
    if not comparison_seasons:
        comparison_seasons = sorted(
            map(int, subset["Season"].unique())
        )

    candidate_records = []
    season_tables: dict[tuple[str, str], pd.DataFrame] = {}
    for (
        architecture,
        model,
    ), group in subset.loc[
        subset["Season"].isin(comparison_seasons)
    ].groupby(["Architecture", "Model"], observed=True):
        observed_seasons = set(
            map(int, group["Season"].unique())
        )
        if observed_seasons != set(comparison_seasons):
            continue
        season_scores = season_metric_values(group, "Prediction")
        mean_brier = float(season_scores["Brier"].mean())
        std_brier = (
            float(season_scores["Brier"].std(ddof=1))
            if len(season_scores) > 1
            else 0.0
        )
        candidate_records.append(
            {
                "Gender": gender,
                "Architecture": str(architecture),
                "Model": str(model),
                "ComparisonSeasons": json.dumps(
                    comparison_seasons
                ),
                "Seasons": len(comparison_seasons),
                "MacroSeasonBrier": mean_brier,
                "SeasonBrierStd": std_brier,
                "StandardError": (
                    std_brier / math.sqrt(len(season_scores))
                    if len(season_scores) > 1
                    else 0.0
                ),
                "ComplexityRank": stream_complexity(
                    str(architecture), str(model)
                ),
            }
        )
        season_tables[(str(architecture), str(model))] = (
            season_scores
        )

    candidates = pd.DataFrame(candidate_records).sort_values(
        ["MacroSeasonBrier", "ComplexityRank"]
    ).reset_index(drop=True)
    assert not candidates.empty
    best = candidates.iloc[0]
    threshold = float(best["MacroSeasonBrier"]) + float(
        best["StandardError"]
    )
    eligible = candidates.loc[
        candidates["MacroSeasonBrier"] <= threshold + 1e-12
    ].sort_values(["ComplexityRank", "MacroSeasonBrier"])
    chosen = eligible.iloc[0]
    candidates["OneSEThreshold"] = threshold
    candidates["WithinOneSE"] = (
        candidates["MacroSeasonBrier"] <= threshold + 1e-12
    )
    candidates["Selected"] = (
        candidates["Architecture"].eq(chosen["Architecture"])
        & candidates["Model"].eq(chosen["Model"])
    )
    selection = {
        "gender": gender,
        "architecture": str(chosen["Architecture"]),
        "model": str(chosen["Model"]),
        "comparison_seasons": comparison_seasons,
        "macro_season_brier": float(
            chosen["MacroSeasonBrier"]
        ),
        "one_se_threshold": threshold,
        "complexity_rank": int(chosen["ComplexityRank"]),
    }
    return selection, candidates


RECIPE_SELECTIONS = {}
recipe_candidate_tables = []
for gender in ("M", "W"):
    selection, candidates = select_stream_one_se(
        development_oof,
        gender=gender,
    )
    RECIPE_SELECTIONS[gender] = selection
    recipe_candidate_tables.append(candidates)

recipe_candidates = pd.concat(
    recipe_candidate_tables, ignore_index=True
)
recipe_candidates.to_csv(
    MODEL_REPORTS / f"recipe_candidates_{RUN_MODE}.csv",
    index=False,
)

recipe_status = (
    "frozen_development_recipe"
    if RUN_MODE in {"standard", "exhaustive"}
    else "smoke_only_not_frozen"
)
MODEL_RECIPE = {
    "recipe_version": 1,
    "status": recipe_status,
    "created_from_run_mode": RUN_MODE,
    "source_contracts": EXPECTED_CONTRACTS,
    "locked_benchmark_evaluated": False,
    "locked_benchmark_seasons": sorted(LOCKED_SEASONS),
    "primary_selections": {
        "men": RECIPE_SELECTIONS["M"],
        "women": RECIPE_SELECTIONS["W"],
    },
    "training_procedure": {
        "feature_selector": "block_aware_season_stability_v1",
        "hyperparameter_search": "nested_optuna_tpe",
        "boosting_rounds": "median_inner_best_iteration",
        "calibration": "cross_fitted_by_inner_season_one_se_rule",
        "ensemble": "nonnegative_simplex_with_redundancy_pruning",
        "separate_primary": True,
        "pooled_challenger": True,
        "partial_pooling": True,
    },
    "seed_free_fallback": {
        "purpose": (
            "valid probabilities for all required hypothetical "
            "Stage 2 pairs; actual scored tournament teams are seeded"
        ),
        "candidate_sets": {
            "men": "men_rich_seed_free",
            "women": "women_rich_seed_free",
            "pooled": "pooled_rich_seed_free",
        },
        "model_family": "elastic_logistic_and_xgb_challenger",
        "selection_rule": "same nested procedure without seed-derived fields",
    },
    "resource_guards": {
        "linear_feature_cap": MODEL_CONFIG["feature_selection"][
            "linear_feature_cap"
        ],
        "tree_feature_cap": MODEL_CONFIG["feature_selection"][
            "tree_feature_cap"
        ],
        "margin_feature_cap": MODEL_CONFIG["feature_selection"][
            "margin_feature_cap"
        ],
        "maximum_threads": MAX_THREADS,
        "stage2_loaded_during_selection": False,
    },
}
MODEL_RECIPE["recipe_sha256"] = object_sha256(MODEL_RECIPE)

recipe_filename = (
    "model_recipe_v1.yaml"
    if recipe_status == "frozen_development_recipe"
    else "model_recipe_smoke.yaml"
)
recipe_path = CONFIG_DIR / recipe_filename
recipe_path.write_text(
    yaml.safe_dump(MODEL_RECIPE, sort_keys=False),
    encoding="utf-8",
)
(MODEL_REPORTS / recipe_filename).write_text(
    yaml.safe_dump(MODEL_RECIPE, sort_keys=False),
    encoding="utf-8",
)

print(yaml.safe_dump(MODEL_RECIPE, sort_keys=False))
recipe_candidates


In [ ]:
# Paired comparisons of the chosen stream against the seed baseline
# and best standalone rich model on exactly matching games.
paired_records = []
for gender, selection in RECIPE_SELECTIONS.items():
    chosen = development_oof.loc[
        development_oof["Gender"].eq(gender)
        & development_oof["Universe"].eq("rich")
        & development_oof["Architecture"].eq(
            selection["architecture"]
        )
        & development_oof["Model"].eq(selection["model"])
        & development_oof["Season"].isin(
            selection["comparison_seasons"]
        )
    ][["TargetKey", "Season", "Team1Win", "Prediction"]].rename(
        columns={"Prediction": "ChosenPrediction"}
    )

    comparison_candidates = recipe_candidates.loc[
        recipe_candidates["Gender"].eq(gender)
        & recipe_candidates["Model"].eq("seed_logistic")
    ].sort_values("MacroSeasonBrier")
    if not comparison_candidates.empty:
        seed_row = comparison_candidates.iloc[0]
        seed = development_oof.loc[
            development_oof["Gender"].eq(gender)
            & development_oof["Universe"].eq("rich")
            & development_oof["Architecture"].eq(
                seed_row["Architecture"]
            )
            & development_oof["Model"].eq("seed_logistic")
            & development_oof["Season"].isin(
                selection["comparison_seasons"]
            )
        ][["TargetKey", "Prediction"]].rename(
            columns={"Prediction": "SeedPrediction"}
        )
        aligned = chosen.merge(
            seed, on="TargetKey", how="inner", validate="one_to_one"
        )
        if not aligned.empty:
            result = season_cluster_bootstrap_difference(
                aligned,
                probability_a="ChosenPrediction",
                probability_b="SeedPrediction",
                repetitions=int(MODE["bootstrap_repetitions"]),
                seed=SEED + (0 if gender == "M" else 1),
            )
            paired_records.append(
                {
                    "Gender": gender,
                    "ModelA": (
                        f"{selection['architecture']}/"
                        f"{selection['model']}"
                    ),
                    "ModelB": (
                        f"{seed_row['Architecture']}/seed_logistic"
                    ),
                    "Rows": len(aligned),
                    "Seasons": aligned["Season"].nunique(),
                    **result,
                }
            )

paired_comparisons = pd.DataFrame(paired_records)
paired_comparisons.to_csv(
    MODEL_REPORTS / f"paired_season_bootstrap_{RUN_MODE}.csv",
    index=False,
)
paired_comparisons


## 18. Bounded explainability after recipe selection

Explainability is deliberately **post-selection** so it cannot become an informal way to tune against the final development years.

For each gender, the notebook reconstructs the latest rich outer-fold XGBoost model and produces:

- mean absolute SHAP importance;
- SHAP beeswarm plot on a bounded held-out sample;
- held-out feature permutation loss increase;
- selected-feature and block context.

These explanations describe one historically legal deployment model. They do not claim causality, and they do not inspect the locked benchmark.


In [ ]:
EXPLAINABILITY_RECORDS: list[dict[str, Any]] = []
if MODE["run_explainability"]:
    for gender in ("M", "W"):
        context = outer_plan.loc[
            outer_plan["Universe"].eq("rich")
            & outer_plan["Architecture"].eq("separate_gender")
            & outer_plan["Gender"].eq(gender)
        ].sort_values("ValidationSeason")
        if context.empty:
            continue
        outer = context.iloc[-1]
        outer_fold_id = str(outer["OuterFoldID"])
        directory = outer_model_directory(
            outer_fold_id, "xgb_classifier"
        )
        metadata_path = directory / "metadata.json"
        if not metadata_path.exists():
            warnings.warn(
                f"No XGBoost checkpoint for explainability: "
                f"{outer_fold_id}"
            )
            continue

        try:
            metadata = read_json(metadata_path)
            features = list(metadata["outer_selected_features"])
            params = dict(metadata["parameters"])
            fixed_iterations = metadata.get(
                "fixed_iterations_from_inner"
            )
            training_seasons = parse_json_int_list(
                outer["TrainingSeasonsJSON"]
            )
            validation_season = int(outer["ValidationSeason"])
            training = rows_for_context(
                development,
                seasons=training_seasons,
                gender=gender,
                universe="rich",
            )
            validation = rows_for_context(
                development,
                seasons=[validation_season],
                gender=gender,
                universe="rich",
            )
            spec = MODEL_SPECS["xgb_classifier"]
            fitted, raw = fit_predictor(
                spec,
                params,
                training,
                validation,
                features,
                fixed_iterations=(
                    int(fixed_iterations)
                    if fixed_iterations is not None
                    else None
                ),
                random_seed=SEED + validation_season,
            )
            assert raw is not None

            sample = validation.sample(
                n=min(int(MODE["shap_rows"]), len(validation)),
                random_state=SEED,
            ).copy()
            x_sample_matrix = prepare_numeric_matrix(sample, features)

            # XGBoost 3 serializes base_score as a vector-valued JSON field. Some
            # SHAP TreeExplainer releases still attempt to parse that field as a
            # scalar. Use XGBoost's native exact TreeSHAP implementation instead.
            # Contributions are in raw-margin (log-odds) space; permutation
            # importance below evaluates the final symmetry-enforced probability.
            sample_dmatrix = xgb.DMatrix(
                x_sample_matrix,
                feature_names=features,
            )
            iteration = (
                int(fitted.best_iteration)
                if fitted.best_iteration is not None
                else int(fitted.model.num_boosted_rounds())
            )
            contribution_matrix = np.asarray(
                fitted.model.predict(
                    sample_dmatrix,
                    pred_contribs=True,
                    iteration_range=(0, max(1, iteration)),
                    strict_shape=False,
                ),
                dtype=np.float64,
            )
            assert contribution_matrix.ndim == 2
            assert contribution_matrix.shape[1] == len(features) + 1
            values = contribution_matrix[:, :-1]
            base_values = contribution_matrix[:, -1]
            shap_values = shap.Explanation(
                values=values,
                base_values=base_values,
                data=x_sample_matrix,
                feature_names=features,
            )
            mean_abs = np.mean(np.abs(values), axis=0)
            shap_table = pd.DataFrame(
                {
                    "Gender": gender,
                    "OuterFoldID": outer_fold_id,
                    "ValidationSeason": validation_season,
                    "Feature": features,
                    "Block": [
                        FEATURE_TO_BLOCK.get(feature, "other")
                        for feature in features
                    ],
                    "MeanAbsoluteSHAP": mean_abs,
                }
            ).sort_values("MeanAbsoluteSHAP", ascending=False)
            shap_table.to_csv(
                MODEL_REPORTS
                / f"xgb_shap_importance_{gender}_{RUN_MODE}.csv",
                index=False,
            )

            plt.figure(figsize=(10, 8))
            shap.plots.beeswarm(
                shap_values,
                max_display=25,
                show=False,
            )
            plt.title(
                f"{gender}: held-out {validation_season} "
                "XGBoost SHAP summary"
            )
            plt.tight_layout()
            plt.savefig(
                FIGURE_DIR
                / f"xgb_shap_beeswarm_{gender}_{RUN_MODE}.png",
                dpi=180,
                bbox_inches="tight",
            )
            plt.show()

            # Bounded held-out permutation analysis on top SHAP features.
            baseline_brier = float(
                np.mean(
                    (
                        clip_probability(raw)
                        - validation["Team1Win"].to_numpy(dtype=float)
                    )
                    ** 2
                )
            )
            rng = np.random.default_rng(SEED)
            permutation_records = []
            for feature in shap_table.head(30)["Feature"]:
                permuted = validation.copy()
                values_to_shuffle = permuted[feature].to_numpy(copy=True)
                rng.shuffle(values_to_shuffle)
                permuted[feature] = values_to_shuffle
                permuted_probability = clip_probability(
                    fitted.predict_raw(permuted)
                )
                permuted_brier = float(
                    np.mean(
                        (
                            permuted_probability
                            - permuted["Team1Win"].to_numpy(dtype=float)
                        )
                        ** 2
                    )
                )
                permutation_records.append(
                    {
                        "Gender": gender,
                        "OuterFoldID": outer_fold_id,
                        "ValidationSeason": validation_season,
                        "Feature": feature,
                        "Block": FEATURE_TO_BLOCK.get(
                            feature, "other"
                        ),
                        "BaselineRawBrier": baseline_brier,
                        "PermutedRawBrier": permuted_brier,
                        "BrierIncrease": (
                            permuted_brier - baseline_brier
                        ),
                    }
                )
            permutation_table = pd.DataFrame(
                permutation_records
            ).sort_values("BrierIncrease", ascending=False)
            permutation_table.to_csv(
                MODEL_REPORTS
                / f"xgb_permutation_importance_{gender}_{RUN_MODE}.csv",
                index=False,
            )
            EXPLAINABILITY_RECORDS.append(
                {
                    "Gender": gender,
                    "OuterFoldID": outer_fold_id,
                    "ValidationSeason": validation_season,
                    "RowsExplained": len(sample),
                    "FeaturesExplained": len(features),
                    "TopSHAPFeature": (
                        shap_table.iloc[0]["Feature"]
                        if not shap_table.empty
                        else None
                    ),
                    "TopPermutationFeature": (
                        permutation_table.iloc[0]["Feature"]
                        if not permutation_table.empty
                        else None
                    ),
                    "AttributionEngine": "xgboost_native_pred_contribs",
                    "AttributionSpace": "raw_margin_log_odds",
                    "Status": "complete",
                }
            )
            del (
                training,
                validation,
                fitted,
                sample,
                x_sample_matrix,
                sample_dmatrix,
                contribution_matrix,
                shap_values,
            )
            gc.collect()
        except Exception as exc:
            EXPLAINABILITY_RECORDS.append(
                {
                    "Gender": gender,
                    "OuterFoldID": outer_fold_id,
                    "Status": "failed",
                    "Error": repr(exc),
                }
            )
            print("EXPLAINABILITY FAILED:", gender, repr(exc))

explainability_summary = pd.DataFrame(EXPLAINABILITY_RECORDS)
explainability_summary.to_csv(
    MODEL_REPORTS / f"explainability_summary_{RUN_MODE}.csv",
    index=False,
)
explainability_summary


## 19. Final integrity checks and development-stage artifact manifest

The notebook is complete only when:

- development predictions contain no locked seasons;
- all predictions are finite probabilities;
- feature caps are respected;
- ensemble weights are nonnegative and sum to one;
- the frozen recipe says the benchmark was not evaluated;
- every required report and OOF artifact exists;
- core model failures are surfaced explicitly.

`smoke` mode validates machinery but does not freeze the final recipe. A complete `standard` or `exhaustive` run is required before notebook `04`.


In [ ]:
FINAL_CHECKS: list[dict[str, Any]] = []


def add_final_check(
    name: str,
    passed: bool,
    details: str,
    *,
    blocking: bool = True,
) -> None:
    FINAL_CHECKS.append(
        {
            "Check": name,
            "Passed": bool(passed),
            "Blocking": bool(blocking),
            "Details": details,
        }
    )


add_final_check(
    "development rows only",
    development["DatasetRole"].eq("development").all(),
    f"rows={len(development):,}",
)
add_final_check(
    "locked seasons absent from loaded model-selection rows",
    not development["Season"].isin(LOCKED_SEASONS).any(),
    f"locked={sorted(LOCKED_SEASONS)}",
)
add_final_check(
    "OOF predictions exclude locked seasons",
    not development_oof["Season"].isin(LOCKED_SEASONS).any(),
    f"max season={development_oof['Season'].max()}",
)
add_final_check(
    "OOF probabilities finite and bounded",
    np.isfinite(
        development_oof["Prediction"].to_numpy(dtype=float)
    ).all()
    and development_oof["Prediction"].between(0, 1).all(),
    f"rows={len(development_oof):,}",
)
add_final_check(
    "OOF model keys unique",
    development_oof.duplicated(
        ["OuterFoldID", "Model", "TargetKey"]
    ).sum()
    == 0,
    "OuterFoldID + Model + TargetKey",
)
add_final_check(
    "source feature store passed notebook 02",
    READINESS_02["blocking_feature_check_failures"] == 0
    and READINESS_02["symmetry_failures"] == 0,
    (
        f"feature checks={READINESS_02['blocking_feature_checks']}; "
        f"symmetry checks={READINESS_02['symmetry_checks']}"
    ),
)
add_final_check(
    "model recipe did not evaluate benchmark",
    MODEL_RECIPE["locked_benchmark_evaluated"] is False,
    MODEL_RECIPE["status"],
)
add_final_check(
    "Stage 2 matrix not loaded in selection notebook",
    "stage2_matchups" not in globals(),
    f"Stage2 file remains on disk: {STAGE2_STORE_PATH}",
)
add_final_check(
    "feature counts respect hard run-mode cap",
    (
        development_oof.loc[
            development_oof["FeatureCount"].notna(),
            "FeatureCount",
        ]
        <= max(
            int(MODE["linear_feature_cap"]),
            int(MODE["tree_feature_cap"]),
            int(MODE["margin_feature_cap"]),
        )
    ).all(),
    json.dumps(
        {
            "linear": MODE["linear_feature_cap"],
            "tree": MODE["tree_feature_cap"],
            "margin": MODE["margin_feature_cap"],
        }
    ),
)

if not ensemble_weights.empty:
    weight_sums = ensemble_weights.groupby(
        "OuterFoldID", observed=True
    )["Weight"].sum()
    add_final_check(
        "ensemble weights nonnegative",
        ensemble_weights["Weight"].ge(-1e-12).all(),
        f"minimum={ensemble_weights['Weight'].min():.8f}",
    )
    add_final_check(
        "ensemble weights sum to one",
        np.allclose(weight_sums.to_numpy(), 1.0, atol=1e-7),
        (
            f"min sum={weight_sums.min():.8f}; "
            f"max sum={weight_sums.max():.8f}"
        ),
    )
else:
    add_final_check(
        "ensemble output available",
        False,
        "No ensemble weights produced.",
        blocking=RUN_MODE != "smoke",
    )

core_rich_models = {
    "elastic_logistic",
    "xgb_classifier",
    "lgb_classifier",
    "ridge_margin",
    "xgb_margin",
}
rich_completed_models = set(
    development_oof.loc[
        development_oof["Universe"].eq("rich"), "Model"
    ]
)
missing_core_rich = sorted(
    core_rich_models.difference(rich_completed_models)
)
add_final_check(
    "core rich model families completed",
    not missing_core_rich,
    f"missing={missing_core_rich}",
    blocking=RUN_MODE != "smoke",
)

# Diagnostic completeness is mandatory in standard/exhaustive mode.
ablation_failure_records = [
    record
    for record in MODEL_FAILURES
    if str(record.get("Model", "")).startswith("ablation__")
]
add_final_check(
    "feature-block ablation completed without failures",
    len(ablation_failure_records) == 0,
    f"failures={len(ablation_failure_records)}",
    blocking=RUN_MODE != "smoke" and bool(MODE["run_ablation"]),
)

expected_explainability_genders = {"M", "W"} if MODE["run_explainability"] else set()
completed_explainability_genders = (
    set(
        explainability_summary.loc[
            explainability_summary.get("Status", pd.Series(dtype=str)).eq("complete"),
            "Gender",
        ].astype(str)
    )
    if not explainability_summary.empty and "Status" in explainability_summary.columns
    else set()
)
add_final_check(
    "held-out explainability completed for both genders",
    expected_explainability_genders.issubset(completed_explainability_genders),
    (
        f"expected={sorted(expected_explainability_genders)}; "
        f"completed={sorted(completed_explainability_genders)}"
    ),
    blocking=RUN_MODE != "smoke" and bool(MODE["run_explainability"]),
)

add_final_check(
    "recipe frozen only after full run",
    (
        MODEL_RECIPE["status"] == "frozen_development_recipe"
        if RUN_MODE in {"standard", "exhaustive"}
        else MODEL_RECIPE["status"] == "smoke_only_not_frozen"
    ),
    MODEL_RECIPE["status"],
)

final_checks = pd.DataFrame(FINAL_CHECKS)
final_checks.to_csv(
    MODEL_REPORTS / f"final_checks_{RUN_MODE}.csv",
    index=False,
)
blocking_failures = final_checks.loc[
    final_checks["Blocking"] & ~final_checks["Passed"]
]

metadata_records = []
calibration_files = []
for metadata_path in (
    CACHE_DIR / "outer_models"
).glob("*/*/metadata.json"):
    try:
        metadata_records.append(read_json(metadata_path))
    except Exception:
        continue
for calibration_path in (
    CACHE_DIR / "outer_models"
).glob("*/*/calibration_audit.csv"):
    try:
        calibration_files.append(pd.read_csv(calibration_path))
    except Exception:
        continue

metadata_table = pd.json_normalize(metadata_records)
metadata_table.to_csv(
    MODEL_REPORTS / f"all_model_metadata_{RUN_MODE}.csv",
    index=False,
)
all_calibration_audits = (
    pd.concat(calibration_files, ignore_index=True)
    if calibration_files
    else pd.DataFrame()
)
all_calibration_audits.to_csv(
    MODEL_REPORTS / f"all_calibration_audits_{RUN_MODE}.csv",
    index=False,
)

artifact_paths = {
    "development_oof": oof_path,
    "leaderboard": (
        MODEL_REPORTS
        / f"development_leaderboard_{RUN_MODE}.csv"
    ),
    "season_metrics": (
        MODEL_REPORTS
        / f"development_brier_by_season_{RUN_MODE}.csv"
    ),
    "recipe": recipe_path,
    "final_checks": (
        MODEL_REPORTS / f"final_checks_{RUN_MODE}.csv"
    ),
}
if not ablation_oof.empty:
    artifact_paths["ablation_oof"] = (
        PROCESSED
        / f"development_feature_ablation_oof_{RUN_MODE}.parquet"
    )

artifact_manifest = pd.DataFrame(
    [
        {
            "Artifact": name,
            "Path": str(path),
            "Exists": path.exists(),
            "SizeMB": (
                round(path.stat().st_size / 1024**2, 3)
                if path.exists()
                else np.nan
            ),
            "SHA256": (
                sha256_file(path) if path.exists() else None
            ),
            "SplitContractSHA256": SPLITS["contract_sha256"],
            "FeatureContractSHA256": FEATURE_CONFIG[
                "feature_contract_sha256"
            ],
            "ModelContractSHA256": MODEL_CONFIG[
                "model_contract_sha256"
            ],
        }
        for name, path in artifact_paths.items()
    ]
)
artifact_manifest.to_csv(
    MODEL_REPORTS / f"artifact_manifest_{RUN_MODE}.csv",
    index=False,
)

readiness = {
    "notebook": "03_nested_oof_model_laboratory.ipynb",
    "status": (
        "complete"
        if blocking_failures.empty
        else "blocking_failures"
    ),
    "run_mode": RUN_MODE,
    "contracts": EXPECTED_CONTRACTS,
    "development_rows": int(len(development)),
    "development_oof_rows": int(len(development_oof)),
    "outer_folds_planned": int(
        outer_plan["OuterFoldID"].nunique()
    ),
    "models_completed": int(
        development_oof["Model"].nunique()
    ),
    "prediction_streams": int(
        development_oof[
            ["Architecture", "Universe", "Model"]
        ].drop_duplicates().shape[0]
    ),
    "ensemble_folds": int(
        development_oof["Model"]
        .eq("constrained_ensemble")
        .groupby(development_oof["OuterFoldID"])
        .any()
        .sum()
    ),
    "partial_pooling_rows": int(
        development_oof["Model"]
        .eq("partial_pooling_ensemble")
        .sum()
    ),
    "model_failures": int(len(MODEL_FAILURES)),
    "ablation_failures": int(len(ablation_failure_records)),
    "explainability_completed_genders": sorted(
        completed_explainability_genders
    ),
    "diagnostic_patch_version": 2,
    "blocking_final_check_failures": int(
        len(blocking_failures)
    ),
    "recipe_status": MODEL_RECIPE["status"],
    "recipe_sha256": MODEL_RECIPE["recipe_sha256"],
    "selected_men": RECIPE_SELECTIONS["M"],
    "selected_women": RECIPE_SELECTIONS["W"],
    "maximum_observed_feature_count": (
        int(
            development_oof.loc[
                development_oof["FeatureCount"].notna(),
                "FeatureCount",
            ].max()
        )
        if development_oof["FeatureCount"].notna().any()
        else 0
    ),
    "peak_logged_rss_mb": (
        float(resource_log["EndRSSMB"].max())
        if not resource_log.empty
        else float(rss_mb())
    ),
    "stage2_loaded": False,
    "locked_benchmark_evaluated": False,
    "artifacts": {
        row["Artifact"]: row["SHA256"]
        for _, row in artifact_manifest.iterrows()
    },
}
atomic_write_json(
    MODEL_REPORTS / f"03_readiness_summary_{RUN_MODE}.json",
    readiness,
)

protocol = f'''# Nested OOF Model Laboratory Protocol

## Contracts

- Split: `{SPLITS["contract_sha256"]}`
- Feature: `{FEATURE_CONFIG["feature_contract_sha256"]}`
- Model: `{MODEL_CONFIG["model_contract_sha256"]}`
- Run mode: `{RUN_MODE}`

## Development boundary

- Development targets end in {DEVELOPMENT_LAST_SEASON}.
- Locked benchmark seasons {sorted(LOCKED_SEASONS)} were not loaded for model selection.
- Stage 2 was not loaded.
- Every outer model uses training seasons strictly earlier than its validation season.
- Early stopping and boosting length come from inner folds only.

## Dimensionality control

- Fold-fitted missingness and variance filtering.
- Season-stability scoring.
- Feature-block quotas.
- Correlation pruning at {MODEL_CONFIG["feature_selection"]["correlation_prune_threshold"]}.
- Adaptive caps by model family and training-row count.
- `float32` projected matrices and sequential fitting.
- Pre-fit RAM headroom checks and resumable atomic checkpoints.
- Cross-fold feature-selection and standardized-coefficient stability audits.

## Calibration and ensembles

- Calibration is selected with cross-fitted inner-season predictions.
- Ensemble weights are nonnegative, sum to one, and are learned from inner OOF only.
- Men/women separate systems are primary.
- Pooled and partial-pooling systems are challengers.

## Benchmark boundary

Notebook 04 is the first stage permitted to load and score 2022–2025 labels.
'''
(MODEL_REPORTS / "MODEL_LAB_PROTOCOL.md").write_text(
    protocol,
    encoding="utf-8",
)

print(json.dumps(readiness, indent=2))
display(final_checks)

if not blocking_failures.empty:
    display(blocking_failures)
    raise AssertionError(
        "Notebook 03 has blocking failures. Review final_checks and "
        "model_failures before proceeding."
    )

print(
    "\nNOTEBOOK 03 COMPLETE — nested development OOF modeling, "
    "calibration, dimensionality control, and architecture comparison "
    "are complete."
)
if RUN_MODE == "smoke":
    print(
        "Smoke mode validated the machinery. Re-run with "
        "MM_RUN_MODE=standard before notebook 04."
    )
else:
    print(
        "The development recipe is frozen. The locked 2022–2025 "
        "benchmark remains unopened."
    )
print(
    "Next: notebook 04 — locked benchmark, robustness adjudication, "
    "final 2026 refit, chunked Stage 2 scoring, and submission audit."
)


# Stop here only after the v2 diagnostic checks pass and return the executed notebook for review

The next notebook will be:

```text
04_locked_benchmark_and_2026_submission.ipynb
```

It will:

1. load the frozen `model_recipe_v1.yaml`;
2. evaluate the prequential and static-block protocols on locked 2022–2025;
3. refuse all post-benchmark retuning;
4. adjudicate calibration, drift, and failure slices;
5. refit the frozen recipe through 2025;
6. load Stage 2 in projected chunks rather than materializing its full 1,000-feature matrix;
7. route men and women to the correct system;
8. train a documented seed-free fallback for unseeded hypothetical pairs;
9. validate probability symmetry, order, bounds, missingness, and submission schema;
10. write the final reproducible submission and employer-facing model card.

Do not manually edit the recipe after viewing locked results. Any scientifically justified revision must create a new, explicitly versioned protocol and treat the old benchmark as consumed.
